# RAG Pipeline — Data Prep (Load → Chunk → Embed → Chroma → Persist)

**Input:** standardized JSON files already produced by `collecting_data.ipynb` + `document_processing.ipynb`
(stored in Google Drive). Each JSON follows this schema:

```json
{
  "document_id": "...",
  "title": "...",
  "source_file": "...",
  "document_type": "json | pdf | html | image",
  "language": "en",
  "pages": [
    {
      "page": 1,
      "blocks": [
        {"type": "heading", "text": "..."},
        {"type": "paragraph", "text": "..."}
      ]
    }
  ],
  "metadata": {
    "document_id": "...",
    "title": "...",
    "source_url": "...",
    "document_type": "...",
    "language": "en",
    "page_count": 1,
    "crawl_category": "...",
    "domain": "catalog.mit.edu",
    "sha256": "...",
    "word_count": 78,
    ...
  }
}
```

This notebook covers **Phase 2, sections 2.1–2.3 (+2.7 export)** of the graduation project spec:

1. **Load & Inspect** — count docs/pages/blocks, spot missing or malformed data
2. **Chunking** — split into overlapping, metadata-tagged chunks
3. **Embeddings** — encode chunks with a sentence-transformers model
4. **Chroma Vector DB** — store chunks + embeddings + metadata
5. **Persist** — save the Chroma store + config to disk so the FastAPI backend can load it without rebuilding

> Run top-to-bottom in Colab (**Kernel → Restart & Run All** should work cleanly).

---

**Revision note:** Sections 0-6 (load, chunk, embed, Chroma, persist, retrieve, retrieval eval)
are unchanged from the previous run and keep their existing outputs below. Section 7's Ollama
install/health-check was hardened (it previously left a broken `llama-server` binary in place
after an interrupted download, which made every downstream generation and evaluation cell
report false 0% results) and Sections 7-9's outputs were cleared since they need a fresh
top-to-bottom run to produce a real, non-stale result. Section 9 (Export / Backend-Readiness
Verification) is new.


## 0. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip -q install chromadb sentence-transformers tqdm pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [3]:
import os, json, glob, hashlib, uuid, re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm

JSON_DIR = "/content/drive/MyDrive/SmartUniversityAssistant/data/processed/documents"
VECTOR_STORE_DIR = "/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store"
CONFIG_PATH = os.path.join(VECTOR_STORE_DIR, "config.json")

os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

json_paths = sorted(glob.glob(os.path.join(JSON_DIR, "**", "*.json"), recursive=True))
print(f"Found {len(json_paths)} JSON files under {JSON_DIR}")


Found 1264 JSON files under /content/drive/MyDrive/SmartUniversityAssistant/data/processed/documents


## 1. Load & Inspect (Phase 2.1)

Loads every JSON file, walks `pages -> blocks`, and reports:
- how many documents / pages / blocks
- block type distribution
- which files failed to parse or look empty/malformed
- basic language / word-count stats from `metadata`


In [4]:
def load_doc(path):
    """Load one standardized JSON file. Returns (doc_dict, error_str_or_None)."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data, None
    except Exception as e:
        return None, str(e)

def extract_block_text(block):
    """Best-effort text extraction for any block shape we might encounter
    (heading/paragraph have 'text'; tables/lists may use 'rows'/'items')."""
    if not isinstance(block, dict):
        return str(block)
    if "text" in block and block["text"]:
        return str(block["text"]).strip()
    if "rows" in block:  # table-like block
        rows = block["rows"]
        return "\n".join(" | ".join(str(c) for c in row) for row in rows)
    if "items" in block:  # list-like block
        return "\n".join(f"- {item}" for item in block["items"])
    return ""


In [5]:
docs = []
parse_errors = []

for p in tqdm(json_paths, desc="Loading JSON files"):
    data, err = load_doc(p)
    if err:
        parse_errors.append({"file": p, "error": err})
        continue
    docs.append(data)

print(f"Loaded {len(docs)} documents successfully")
print(f"Failed to parse: {len(parse_errors)} files")
if parse_errors:
    display(pd.DataFrame(parse_errors).head(20))


Loading JSON files:   0%|          | 0/1264 [00:00<?, ?it/s]

Loaded 1264 documents successfully
Failed to parse: 0 files


In [6]:
total_pages = 0
total_blocks = 0
block_type_counter = Counter()
empty_block_count = 0
lang_counter = Counter()
doctype_counter = Counter()
word_counts = []
docs_with_no_pages = []
docs_with_no_blocks = []

for d in docs:
    pages = d.get("pages", [])
    total_pages += len(pages)
    lang_counter[d.get("language", "unknown")] += 1
    doctype_counter[d.get("document_type", "unknown")] += 1

    if not pages:
        docs_with_no_pages.append(d.get("document_id"))
        continue

    doc_has_block = False
    for page in pages:
        blocks = page.get("blocks", [])
        total_blocks += len(blocks)
        for b in blocks:
            block_type_counter[b.get("type", "unknown")] += 1
            txt = extract_block_text(b)
            if not txt:
                empty_block_count += 1
            else:
                doc_has_block = True
    if not doc_has_block:
        docs_with_no_blocks.append(d.get("document_id"))

    meta = d.get("metadata", {})
    if isinstance(meta.get("word_count"), (int, float)):
        word_counts.append(meta["word_count"])

print("=== Load & Inspect summary ===")
print(f"Documents loaded        : {len(docs)}")
print(f"Total pages             : {total_pages}")
print(f"Total blocks            : {total_blocks}")
print(f"Empty/blank blocks      : {empty_block_count}")
print(f"Docs with zero pages    : {len(docs_with_no_pages)}")
print(f"Docs with zero content  : {len(docs_with_no_blocks)}")
print(f"Failed-to-parse files   : {len(parse_errors)}")
print()
print("Document types:", dict(doctype_counter))
print("Languages     :", dict(lang_counter))
print("Block types   :", dict(block_type_counter.most_common()))
if word_counts:
    wc = pd.Series(word_counts)
    print()
    print("word_count (from metadata) stats:")
    print(wc.describe())


=== Load & Inspect summary ===
Documents loaded        : 1264
Total pages             : 2143
Total blocks            : 64253
Empty/blank blocks      : 0
Docs with zero pages    : 0
Docs with zero content  : 122
Failed-to-parse files   : 0

Document types: {'json': 1000, 'image': 126, 'pdf': 136, 'docx': 2}
Languages     : {'en': 1138, 'unknown': 122, 'ur': 1, 'de': 1, 'ar': 1, 'fa': 1}
Block types   : {'paragraph': 46557, 'heading': 12753, 'table': 4943}

word_count (from metadata) stats:
count     1000.000000
mean       684.195000
std       2239.870441
min          2.000000
25%        128.000000
50%        235.000000
75%        584.500000
max      41696.000000
dtype: float64


In [7]:
# Flag docs worth a manual look before chunking
problem_ids = set(docs_with_no_pages) | set(docs_with_no_blocks)
if problem_ids:
    print(f"{len(problem_ids)} document(s) flagged as empty/problematic — excluding them from chunking:")
    for i in list(problem_ids)[:20]:
        print(" -", i)
else:
    print("No empty/problematic documents found.")


122 document(s) flagged as empty/problematic — excluding them from chunking:
 - d3ffbd08955c4399__app_uploads_2026_05_Yoga_in_front_of_Dome_copy_aspect_ratio__7e41f1e69d
 - 10af8a290057bf91__app_uploads_2025_05_43221745990_a607ed2e49_b_aspect_ratio_1__78fa47a6cc
 - 137066791a68ed17__app_uploads_2025_03_7677306000_5840d4a0de_b_1_aspect_ratio_1_fb263e8410
 - 2b4ac5703f6d064e__app_uploads_2025_04_students_in_wellbeing_lab_scaled_aspect__5c39d6d7a8
 - ecfeab19be3a9003__app_uploads_2026_03_srihitha_dasari_mit_00_0_700x0_c_default_6ed03d3b2f
 - 4f7bf5b50ff1a96a__app_uploads_2025_04_student_studying_outside_chapel_copy_2_a_0739de2642
 - f9ba25df8dd50440__app_uploads_2025_04_chapel_exterior_aspect_ratio_1_1_550x0_c_86bb954e56
 - dd33f316dfa3a637__app_uploads_2025_04_DSC_3823_copy_aspect_ratio_1_1_550x0_c_d_2aaa12b979
 - 2ede029df1684614__app_uploads_2025_07_MIT_Vassar_Dining_HORNER1_aspect_ratio_1_05367346ac
 - b76b96722a39b7ab__app_uploads_2025_04_USE_THIS_Suzy_Nelson_101_aspect_ratio_1__925f

## 2. Chunking Strategy (Phase 2.2)

**Updated approach: section-aware chunking, not whole-page concatenation.**

The first version of this notebook concatenated *all* block text on a page into one long
string and then ran a plain sliding word-window over it. That's simple, but it has a real
problem with this dataset: MIT catalog pages routinely pack **several unrelated
subsections** onto a single page (e.g. multiple subjects, or a "Prerequisites" block right
next to an unrelated "Faculty" block). Chunking across the whole page could merge two
unrelated sections into one chunk just because they happened to sit next to each other —
which hurts retrieval precision and makes the "grounded answer" harder to justify.

**What changed:**
1. **Group blocks into sections first.** Within each page, every `heading` block starts a
   new section; the `paragraph`/other blocks that follow belong to that heading until the
   next heading appears. Blocks before the first heading form one leading section. This
   mirrors how the document is actually organized instead of throwing it all into one bag
   of words.
2. **Merge small sections before chunking.** Grouping by heading alone produced a huge
   number of tiny/heading-only chunks whenever a page had many short subsections back to
   back (a heading with one line of text, or a bare heading with nothing under it at all).
   `merge_small_sections()` accumulates consecutive sections on the same page until their
   combined body reaches `SECTION_MERGE_MIN_WORDS`, so a run of short subsections becomes
   one reasonably-sized section instead of several near-empty ones. Any leftover sliver at
   the end of a page is folded into the previous merged section rather than emitted as its
   own tiny chunk. This is a real fix to the chunk-building logic, not a post-hoc filter —
   no content is dropped, it's just grouped more sensibly before the window is applied.
3. **Chunk *within* each (merged) section**, not across sections. A fixed-size, word-count
   sliding window (`CHUNK_SIZE_WORDS` / `CHUNK_OVERLAP_WORDS`, both configurable below) is
   applied to each section's body text. Short sections stay a single chunk; long sections
   split into several overlapping chunks — but a chunk never spans two different pages.
4. **The section heading is re-attached to every chunk cut from that section**, including
   the 2nd/3rd chunk of a long section. When small sections are merged, the first heading
   in the group becomes the chunk's `section_heading`; any headings folded in afterwards
   are kept as plain text inside the chunk body so the information isn't lost.
5. **Page number is taken directly from the page the section came from** and is identical
   for every chunk produced by that section — so there is never an ambiguous or averaged
   page citation.

This is still plain fixed-size + overlap chunking (no semantic/LLM chunking, no extra
dependencies) — the changes are *where the window is applied* (per merged section instead
of per whole page) and *how sections are grouped* (merging small ones) before that window
is applied.

**Chunk size = 220 words, overlap = 40 words (~18%)** — a reasonable size for these short
catalog/subject pages, keeping chunks well under typical embedding context limits while
overlap preserves context cut at a boundary.


In [8]:
# ---- Configurable chunking parameters ----
CHUNK_SIZE_WORDS = 220        # body words per chunk; final word_count includes the re-attached heading
CHUNK_OVERLAP_WORDS = 40      # word overlap between consecutive chunks of the same section
SMALL_CHUNK_WORDS = 10        # threshold used later only for the inspection/QA report, not for filtering
SECTION_MERGE_MIN_WORDS = 60  # sections whose body is below this many words get merged into a
                               # neighboring section (see merge_small_sections) instead of becoming
                               # their own chunk -- this is what actually fixes the tiny/heading-only
                               # chunk explosion, not just the reporting below.

def chunk_words(words, size, overlap):
    """Yield (start_idx, end_idx, chunk_words) with sliding-window overlap."""
    step = size - overlap
    assert step > 0, "overlap must be smaller than chunk size"
    i = 0
    n = len(words)
    while i < n:
        j = min(i + size, n)
        yield i, j, words[i:j]
        if j == n:
            break
        i += step


def group_blocks_into_sections(blocks):
    """Group a page's blocks into sections: a 'heading' block starts a new section, and
    every following non-heading block belongs to that section until the next heading.
    Blocks appearing before the first heading form a leading (headless) section.

    This is what prevents unrelated subsections on the same page from being merged into
    the same chunk just because they are physically close together.
    """
    sections = []
    current = {"heading": None, "body_parts": []}
    started = False
    for b in blocks:
        btype = b.get("type", "")
        txt = extract_block_text(b)
        if not txt:
            continue
        if btype == "heading":
            if started:
                sections.append(current)
            current = {"heading": txt, "body_parts": []}
            started = True
        else:
            current["body_parts"].append(txt)
            started = True
    if started:
        sections.append(current)
    return sections


def _section_word_count(section):
    return len(" ".join(section["body_parts"]).split())


def merge_small_sections(sections, min_words=SECTION_MERGE_MIN_WORDS):
    """Merge adjacent small sections (within the same page) so near-empty and
    heading-only sections don't each turn into their own tiny chunk.

    Sections are accumulated into a running buffer until the buffer's body reaches
    `min_words`, at which point it is flushed as one merged section. Any leftover
    buffer at the end of the page (too small to flush on its own) is folded into the
    last flushed section instead of being emitted as a standalone tiny chunk; if the
    whole page never reaches `min_words`, it stays as a single (unavoidably small)
    section rather than many.

    The first non-empty heading encountered in a merge group becomes that group's
    `section_heading`; headings from sections folded in afterwards are kept as plain
    text inside the body (so the information isn't lost) rather than as a second
    metadata heading.
    """
    if not sections:
        return sections

    merged = []
    buffer = None

    for sec in sections:
        if buffer is None:
            buffer = {"heading": sec["heading"], "body_parts": list(sec["body_parts"])}
        else:
            if not buffer["heading"] and sec["heading"]:
                buffer["heading"] = sec["heading"]
            elif sec["heading"]:
                buffer["body_parts"].append(sec["heading"])
            buffer["body_parts"].extend(sec["body_parts"])

        if _section_word_count(buffer) >= min_words:
            merged.append(buffer)
            buffer = None

    if buffer is not None:
        if merged:
            if buffer["heading"] and buffer["heading"] != merged[-1]["heading"]:
                merged[-1]["body_parts"].append(buffer["heading"])
            merged[-1]["body_parts"].extend(buffer["body_parts"])
        else:
            merged.append(buffer)

    return merged


def build_chunks_for_doc(doc):
    """Section-aware chunking: group blocks per page into heading-anchored sections,
    merge sections that are too small to stand on their own, then slide a fixed-size
    overlapping word window within each (merged) section. The section heading is
    re-attached to every chunk it produces, and every chunk keeps the exact page
    number of the page it came from (never inferred or averaged)."""
    doc_id = doc.get("document_id", "unknown")
    title = doc.get("title", "")
    meta = doc.get("metadata", {})
    source_url = meta.get("source_url", "")
    crawl_category = meta.get("crawl_category", "")
    domain = meta.get("domain", "")
    document_type = doc.get("document_type", meta.get("document_type", "unknown"))
    language = doc.get("language", "unknown")

    chunks = []
    for page in doc.get("pages", []):
        page_num = page.get("page")
        if page_num is None:
            page_num = -1  # sentinel: page number missing from source JSON -> flagged in QA section below

        raw_sections = group_blocks_into_sections(page.get("blocks", []))
        sections = merge_small_sections(raw_sections, SECTION_MERGE_MIN_WORDS)

        for section_idx, section in enumerate(sections):
            heading = section["heading"]
            body_text = "\n\n".join(section["body_parts"]).strip()

            if not body_text and not heading:
                continue  # nothing to chunk

            if not body_text:
                # heading-only section (e.g. a bare title block with no following text
                # anywhere else on the page) -- can still happen if a whole page is just
                # one heading, which merge_small_sections cannot fix any further.
                word_chunks = [heading.split()]
            else:
                body_words = body_text.split()
                word_chunks = [w for (_, _, w) in chunk_words(body_words, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)]

            for chunk_idx, w in enumerate(word_chunks):
                if heading and body_text:
                    chunk_text = (heading + "\n\n" + " ".join(w)).strip()
                else:
                    chunk_text = " ".join(w).strip()
                if not chunk_text:
                    continue

                chunk_id = f"{doc_id}_p{page_num}_s{section_idx}_c{chunk_idx}"
                chunks.append({
                    "id": chunk_id,
                    "text": chunk_text,
                    "metadata": {
                        "document_id": doc_id,
                        "title": title,
                        "page": page_num,
                        "source_url": source_url,
                        "crawl_category": crawl_category,
                        "domain": domain,
                        "document_type": document_type,
                        "language": language,
                        "section_index": section_idx,
                        "section_heading": heading if heading else "",
                        "chunk_index": chunk_idx,
                        "chunk_id": chunk_id,
                        "word_count": len(chunk_text.split()),
                    }
                })
    return chunks


In [9]:
all_chunks = []
for d in tqdm(docs, desc="Chunking documents"):
    if d.get("document_id") in problem_ids:
        continue
    all_chunks.extend(build_chunks_for_doc(d))

print(f"Total chunks created: {len(all_chunks)}")

chunk_lengths = pd.Series([c["metadata"]["word_count"] for c in all_chunks])
print()
print("Chunk word-count distribution:")
print(chunk_lengths.describe())


Chunking documents:   0%|          | 0/1264 [00:00<?, ?it/s]

Total chunks created: 8108

Chunk word-count distribution:
count    8108.000000
mean      194.646769
std        55.720861
min         4.000000
25%       190.000000
50%       225.000000
75%       225.000000
max       244.000000
dtype: float64


### 2.4 Chunk Inspection (Requirement: verify before embedding)

Total count, length distribution, and real examples — including page numbers and headings
preserved with their content.


In [10]:
word_counts_per_chunk = [c["metadata"]["word_count"] for c in all_chunks]

print("=== Chunk statistics ===")
print(f"Total chunks : {len(all_chunks)}")
if word_counts_per_chunk:
    wc = pd.Series(word_counts_per_chunk)
    print(f"Min length   : {wc.min()} words")
    print(f"Max length   : {wc.max()} words")
    print(f"Mean length  : {wc.mean():.1f} words")
    print(f"Median length: {wc.median():.1f} words")
    print()
    print(wc.describe())


=== Chunk statistics ===
Total chunks : 8108
Min length   : 4 words
Max length   : 244 words
Mean length  : 194.6 words
Median length: 225.0 words

count    8108.000000
mean      194.646769
std        55.720861
min         4.000000
25%       190.000000
50%       225.000000
75%       225.000000
max       244.000000
dtype: float64


In [11]:
print("=== A few real chunks with full metadata ===\n")
for c in all_chunks[:3]:
    print("chunk_id:", c["id"])
    print("metadata:", c["metadata"])
    print("text:", c["text"][:300].replace("\\n", " ") + ("..." if len(c["text"]) > 300 else ""))
    print("-" * 80)

print()
print("=== Examples showing page numbers (one chunk per distinct page, first 5 pages seen) ===\n")
seen_pages = {}
for c in all_chunks:
    key = (c["metadata"]["document_id"], c["metadata"]["page"])
    if key not in seen_pages:
        seen_pages[key] = c
    if len(seen_pages) >= 5:
        break
for (doc_id, page), c in seen_pages.items():
    print(f"doc={doc_id} | page={page} | chunk_id={c['id']}")

print()
print("=== Examples where a heading is preserved together with its section's content ===\n")
heading_examples = [c for c in all_chunks if c["metadata"]["section_heading"] and c["metadata"]["word_count"] > 5]
for c in heading_examples[:3]:
    print("heading   :", c["metadata"]["section_heading"])
    print("page      :", c["metadata"]["page"])
    print("chunk text:", c["text"][:250].replace("\\n", " ") + "...")
    print("-" * 80)


=== A few real chunks with full metadata ===

chunk_id: 0029ae65943eda2f_e148e9885c_p1_s0_c0
metadata: {'document_id': '0029ae65943eda2f_e148e9885c', 'title': 'Academic Calendar I MIT Registrar', 'page': 1, 'source_url': 'https://registrar.mit.edu/calendar?f%5B0%5D=category%3A99&f%5B1%5D=student%3A93', 'crawl_category': 'academic_calendar', 'domain': 'registrar.mit.edu', 'document_type': 'json', 'language': 'en', 'section_index': 0, 'section_heading': 'Academic Calendar I MIT Registrar', 'chunk_index': 0, 'chunk_id': '0029ae65943eda2f_e148e9885c_p1_s0_c0', 'word_count': 53}
text: Academic Calendar I MIT Registrar

Academic Calendar September 2026 Search Main Menu Category(-)OrientationThesisAcademic deadlinesDegree list deadlinesTuition & financial aid deadlinesExamsHolidaysMeetings StudentUndergraduate(-)Graduate Year2025-20262026-2027 September 2026 Central graduate studen...
--------------------------------------------------------------------------------
chunk_id: 0040472cc177908d_0

### 2.5 Problematic Chunk Checks

Before moving to embeddings, confirm there are no empty chunks, no chunks with missing
core metadata, and no chunks with ambiguous/missing page numbers.


In [12]:
# Core metadata is required for every chunk. Optional/provenance fields may legitimately be unavailable.
required_meta_keys = [
    "document_id",
    "title",
    "page",
    "document_type",
    "language",
    "section_index",
    "chunk_index",
    "chunk_id",
    "word_count",
]
optional_meta_keys = [
    "source_url",
    "source_file",
    "crawl_category",
    "domain",
    "section_heading",
]

empty_chunks = [c for c in all_chunks if not c["text"].strip()]
tiny_chunks = [c for c in all_chunks if c["metadata"]["word_count"] < SMALL_CHUNK_WORDS]
missing_core_meta_chunks = [
    c for c in all_chunks
    if any(c["metadata"].get(k) in (None, "") for k in required_meta_keys)
]
bad_page_chunks = [
    c for c in all_chunks
    if not isinstance(c["metadata"].get("page"), int) or c["metadata"]["page"] < 0
]
wrong_word_count_chunks = [
    c for c in all_chunks
    if c["metadata"].get("word_count") != len(c["text"].split())
]
duplicate_chunk_ids = len(all_chunks) - len({c["id"] for c in all_chunks})

print(f"Empty chunks                 : {len(empty_chunks)}")
print(f"Extremely small chunks (<{SMALL_CHUNK_WORDS} words): {len(tiny_chunks)}")
print(f"Chunks missing core metadata: {len(missing_core_meta_chunks)}")
print(f"Chunks with missing/invalid page number: {len(bad_page_chunks)}")
print(f"Incorrect word_count values  : {len(wrong_word_count_chunks)}")
print(f"Duplicate chunk IDs         : {duplicate_chunk_ids}")

if tiny_chunks:
    print()
    print("Sample tiny chunks (worth a manual look; usually heading-only sections):")
    for c in tiny_chunks[:5]:
        print(f"  - [{c['metadata']['word_count']} words] {c['id']}: {c['text'][:100]!r}")

if missing_core_meta_chunks:
    print()
    print("Sample chunks with missing core metadata:")
    for c in missing_core_meta_chunks[:5]:
        print(f"  - {c['id']}: {c['metadata']}")

if bad_page_chunks:
    print()
    print("Sample chunks with missing/invalid page number:")
    for c in bad_page_chunks[:5]:
        print(f"  - {c['id']}: page={c['metadata'].get('page')}")

if wrong_word_count_chunks:
    print()
    print("Sample chunks with incorrect word_count:")
    for c in wrong_word_count_chunks[:5]:
        print(f"  - {c['id']}: stored={c['metadata'].get('word_count')}, actual={len(c['text'].split())}")

assert not empty_chunks, "Empty chunks found."
assert not missing_core_meta_chunks, "Core metadata is incomplete."
assert not bad_page_chunks, "Missing/invalid page numbers found."
assert not wrong_word_count_chunks, "word_count does not match final chunk_text."
assert duplicate_chunk_ids == 0, "Duplicate chunk IDs found."

print()
print("Core metadata is complete and internally consistent.")
print("Optional/provenance metadata is allowed to be unavailable.")

Empty chunks                 : 0
Extremely small chunks (<10 words): 7
Chunks missing core metadata: 0
Chunks with missing/invalid page number: 0
Incorrect word_count values  : 0
Duplicate chunk IDs         : 0

Sample tiny chunks (worth a manual look; usually heading-only sections):
  - [4 words] 1044735d6cdc9b32__app_uploads_2025_04_Lauryn_McNair_aspect_ratio_1_1_1_550x0_c_e223c92857_p1_s0_c0: 'اققت 4 ق15 JATTBRS'
  - [6 words] 63f53c0f3064176f__schools_engineering_mechanical_engineering_mechanical_engine_b74b88f574_p3_s0_c0: 'Department of Mechanical Engineering I 5'
  - [8 words] 95f941a00e2e953d__degree_charts_mathematics_course_18_mathematics_course_18_ge_5439b6d0e7_p2_s0_c0: 'MATHEMATICS (COURSE 18) 4 I Mathematics (Course 18)'
  - [5 words] b666ea7c23402bb6__degree_charts_mathematics_course_18_mathematics_course_18_ap_b9928f1975_p3_s0_c0: 'Mathematics (Course 18) I 5'
  - [5 words] d7bf7dcf71aa85bc__app_uploads_2025_03_Group_shot_awards_2024_aspect_ratio_1_1__154f9dc494_p1_s0_c

### Why this chunking strategy fits this dataset and RAG retrieval

- **Section-aware, not page-blind**: MIT catalog pages mix several short subsections
  (subjects, prerequisites, staff info, etc.) on one page — chunking within heading-bound
  sections avoids stitching unrelated content into a single retrievable unit.
- **Small sections are merged, not left tiny**: back-to-back short subsections (or bare
  headings with little/no body) are combined until they reach a sensible minimum size
  (`SECTION_MERGE_MIN_WORDS`) before the word window runs, which is what keeps the corpus
  from filling up with hundreds of near-empty, heading-only chunks that add noise without
  adding retrievable value.
- **Heading + content together**: retrieval quality benefits from the heading acting as a
  built-in "title" for the chunk (e.g. a query about "Global Languages" is more likely to
  match a chunk whose text starts with that heading), and it costs nothing extra since it's
  just string concatenation.
- **Fixed page number per chunk**: because chunks never cross a page boundary, every
  citation the backend shows (`document_id` + `page`) is unambiguous — no chunk is ever
  "half from page 3, half from page 4."
- **Still simple**: no semantic/LLM chunking, no extra dependencies — just grouping by an
  existing `type == "heading"` field, merging small groups, and the same word-count sliding
  window as before. `CHUNK_SIZE_WORDS` / `CHUNK_OVERLAP_WORDS` / `SECTION_MERGE_MIN_WORDS`
  remain the knobs to tune per corpus.


## 3. Embeddings (Phase 2.3)

Using `sentence-transformers/all-MiniLM-L6-v2`: fast, 384-dim, strong general-purpose
choice for CPU-only Colab and it's the same model family that will run cheaply inside the
FastAPI backend (no need for a GPU at inference time).

**Important:** the exact same model + preprocessing must be used to embed the user's query
at request time in the backend — the model name is saved to `config.json` in Section 5 for
that reason.


In [13]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine similarity friendly
)

print("Embeddings shape:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

Embeddings shape: (8108, 384)


## 4. Chroma Vector Database

Stores, for every chunk: the embedding, the raw chunk text (as Chroma's `document`), and
the metadata dict built during chunking (document_id, title, source_url, page, etc.).


In [14]:
import chromadb

COLLECTION_NAME = "rag_assistant_chunks"

chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# Fresh collection each run of this notebook (drop + recreate) so re-running stays idempotent
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"embedding_model": EMBEDDING_MODEL_NAME, "hnsw:space": "cosine"},
)


In [15]:
BATCH_SIZE = 256

def clean_metadata(metadata):
    cleaned = {}
    for key, value in metadata.items():
        if value is None:
            continue
        elif isinstance(value, (str, int, float, bool)):
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

# Safety checks before writing
assert len(all_chunks) == len(embeddings), (len(all_chunks), len(embeddings))
assert len({c["id"] for c in all_chunks}) == len(all_chunks), "Duplicate chunk IDs found"

for start in tqdm(
    range(0, len(all_chunks), BATCH_SIZE),
    desc="Writing to Chroma"
):
    end = start + BATCH_SIZE
    batch = all_chunks[start:end]

    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embeddings[start:end].tolist(),
        metadatas=[clean_metadata(c["metadata"]) for c in batch],
    )

final_count = collection.count()
assert final_count == len(all_chunks), (final_count, len(all_chunks))

print(f"Collection '{COLLECTION_NAME}' now has {final_count} chunks")
print("Chroma write verification: PASSED")


Writing to Chroma:   0%|          | 0/32 [00:00<?, ?it/s]

Collection 'rag_assistant_chunks' now has 8108 chunks
Chroma write verification: PASSED


In [16]:
# Quick retrieval smoke test (sanity-check before persisting)
def test_query(question, k=3):
    q_emb = embedder.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    print(f"Q: {question}")
    for i in range(len(res["ids"][0])):
        meta = res["metadatas"][0][i]
        dist = res["distances"][0][i]
        snippet = res["documents"][0][i][:150].replace("\n", " ")
        print(f"  [{i}] dist={dist:.3f} | {meta.get('title')} (p.{meta.get('page')}) -> {snippet}...")
    print()

sample_questions = [
    "What are the prerequisites for this course?",
    "Who is the summer session representative?",
    "Are regular classes offered during the summer term?",
]
for q in sample_questions:
    test_query(q)


Q: What are the prerequisites for this course?
  [0] dist=0.375 | Search Results I MIT Course Catalog (p.1) -> Materials Science and Engineering (Course 3)  Technology (REST) Requirement [can be satisfied by 18.03 or 18.06 and 18.062[J] (if taken under joint nu...
  [1] dist=0.400 | Brain and Cognitive Sciences (Course 9) I MIT Course Catalog (p.1) -> Brain and Cognitive Sciences (Course 9) I MIT Course Catalog  as communication-intensive (CI-H) to fulfill the Communication Requirement.", "8"], ["Re...
  [2] dist=0.400 | Brain and Cognitive Sciences (Course 9) I MIT Course Catalog (p.1) -> Brain and Cognitive Sciences (Course 9) I MIT Course Catalog  as communication-intensive (CI-H) to fulfill the Communication Requirement.", "8"], ["Re...

Q: Who is the summer session representative?
  [0] dist=0.574 | Political Science (Course 17) I MIT Course Catalog (p.1) -> Political Science (Course 17) I MIT Course Catalog  Political Science (Course 17) Summer Session RepresentativeScott Schnyer

## 5. Persist Vector Store (Phase 2.7)

`chromadb.PersistentClient` already writes to disk on every `add()` call, so the store at
`VECTOR_STORE_DIR` is already durable. This section just writes a `config.json` next to it
so the FastAPI backend knows exactly how to load and query it later, with no rebuilding.


In [17]:
config = {
    "vector_store_dir": VECTOR_STORE_DIR,
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dim": int(embeddings.shape[1]),
    "distance_metric": "cosine",
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "section_merge_min_words": SECTION_MERGE_MIN_WORDS,
    "num_source_documents": len(docs) - len(problem_ids),
    "num_chunks": len(all_chunks),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved config to:", CONFIG_PATH)
print(json.dumps(config, indent=2))


Saved config to: /content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/config.json
{
  "vector_store_dir": "/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store",
  "collection_name": "rag_assistant_chunks",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "distance_metric": "cosine",
  "chunk_size_words": 220,
  "chunk_overlap_words": 40,
  "section_merge_min_words": 60,
  "num_source_documents": 1142,
  "num_chunks": 8108
}


In [18]:
# Verify the persisted Chroma store can be reopened successfully
reopened_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
reopened_collection = reopened_client.get_collection(COLLECTION_NAME)

reopened_count = reopened_collection.count()
print("Persisted collection count:", reopened_count)
print("Expected chunk count:", len(all_chunks))

assert reopened_count == len(all_chunks)
print("Persistence verification: PASSED")


Persisted collection count: 8108
Expected chunk count: 8108
Persistence verification: PASSED


In [19]:
# Final check: list what's on disk — this whole folder is what you copy into backend/data/vector_store/
for root, _, files in os.walk(VECTOR_STORE_DIR):
    for fn in files:
        full = os.path.join(root, fn)
        size_kb = os.path.getsize(full) / 1024
        print(f"{full}  ({size_kb:.1f} KB)")


/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/chroma.sqlite3  (106028.0 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/config.json  (0.4 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/d903d840-0821-4497-a6b1-43844efd6d69/header.bin  (0.1 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/d903d840-0821-4497-a6b1-43844efd6d69/data_level0.bin  (11732.0 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/d903d840-0821-4497-a6b1-43844efd6d69/length.bin  (28.0 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/d903d840-0821-4497-a6b1-43844efd6d69/link_lists.bin  (61.3 KB)
/content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/d903d840-0821-4497-a6b1-43844efd6d69/index_metadata.pickle  (859.5 KB)


## 6. Retrieval (Phase 3)

This section adds the **retrieval step only** on top of the persisted Chroma collection
built above. Nothing in Sections 0-5 (loading, chunking, embedding, Chroma indexing,
persistence) is modified.

`retrieve(question, k=5)` is a small, reusable function that:
1. Embeds the question with the *same* `SentenceTransformer` model used to embed the
   chunks (`EMBEDDING_MODEL_NAME`), so query and document vectors live in the same space.
2. Queries the **reopened, persisted** Chroma collection (`reopened_collection` from
   Section 5) — i.e. it reads from disk the same way the eventual FastAPI backend will,
   not from the in-memory `collection` object.
3. Returns the top-k chunks with their text, document title, page, source, and a
   distance / similarity score, ready to hand to a generation step later.


In [20]:
def retrieve(question, k=5, _collection=None, _embedder=None):
    """Retrieve the top-k most relevant chunks for a natural-language question.

    Parameters
    ----------
    question : str
        The user's natural-language query.
    k : int, default 5
        Number of chunks to retrieve.
    _collection : chromadb Collection, optional
        Defaults to the persisted, reopened collection from Section 5
        (falls back to the in-memory `collection` if that name isn't available,
        so this still works if Section 5's reopen cell was skipped).
    _embedder : SentenceTransformer, optional
        Defaults to the `embedder` created in Section 3 (same model used to embed
        every chunk, so query/document vectors are comparable).

    Returns
    -------
    list[dict]
        One dict per retrieved chunk, each containing:
        rank, chunk_id, text, title, page, source, distance, similarity.
        "source" prefers the crawled source_url, falling back to domain/source_file
        when a URL isn't available. "similarity" = 1 - distance (valid because the
        collection was created with hnsw:space = "cosine").
    """
    coll = _collection or globals().get("reopened_collection") or globals().get("collection")
    emb_model = _embedder or globals().get("embedder")
    if coll is None:
        raise RuntimeError(
            "No Chroma collection available — run Sections 4-5 first "
            "(this creates `collection` and/or `reopened_collection`)."
        )
    if emb_model is None:
        raise RuntimeError(
            "No embedding model available — run Section 3 first (this creates `embedder`)."
        )

    query_embedding = emb_model.encode([question], normalize_embeddings=True).tolist()
    res = coll.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    results = []
    ids = res.get("ids", [[]])[0]
    docs_out = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    for rank, (chunk_id, text, meta, dist) in enumerate(
        zip(ids, docs_out, metas, dists), start=1
    ):
        meta = meta or {}
        source = meta.get("source_url") or meta.get("domain") or meta.get("source_file") or ""
        results.append({
            "rank": rank,
            "chunk_id": chunk_id,
            "text": text,
            "title": meta.get("title", ""),
            "page": meta.get("page"),
            "source": source,
            "distance": float(dist),
            "similarity": 1.0 - float(dist),  # cosine space: similarity = 1 - distance
        })
    return results


# Quick sanity check that the function works end-to-end
_sanity = retrieve("What are the prerequisites for a course?", k=3)
print(f"retrieve() returned {len(_sanity)} results for a sanity-check query.")
for r in _sanity:
    print(f"  [{r['rank']}] sim={r['similarity']:.3f} | {r['title']} (p.{r['page']})")


retrieve() returned 3 results for a sanity-check query.
  [1] sim=0.589 | Search Results I MIT Course Catalog (p.1)
  [2] sim=0.559 | Archaeology and Materials (Course 3-C) I MIT Course Catalog (p.1)
  [3] sim=0.557 | Search Results I MIT Course Catalog (p.1)


### 6.1 Sample Questions (real MIT dataset topics)

The questions below are grounded in what's actually in this corpus (MIT Course Catalog
subject pages such as *Materials Science and Engineering (Course 3)*, *Brain and
Cognitive Sciences (Course 9)*, *Political Science (Course 17)*, plus MIT Registrar
*Academic Calendar* pages covering deadlines, exam periods, and the summer session). The
last two questions are intentionally **out-of-domain** (dining/parking) — this dataset is
course-catalog + registrar content, so those are expected to fail and are kept in on
purpose to produce real failure cases for the evaluation below, rather than only showing
questions the system is guaranteed to answer well.


In [21]:
import pandas as pd

sample_questions = [
    "What are the prerequisites for 18.03?",
    "Who is the Political Science (Course 17) summer session representative?",
    "Are regular classes offered during the MIT summer term?",
    "When does the fourth-quarter Physical Education & Wellness class period begin?",
    "What is the last day to add half-term subjects?",
    "When is the summer session final exam period?",
    "What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?",
    "What subjects fall under Materials Science and Engineering (Course 3)?",
    "What academic deadlines are listed on the MIT Registrar calendar?",
    "What orientation-related events are on the academic calendar?",
    "What are the tuition and financial aid deadlines at MIT?",
    "What is the price of a meal plan at an MIT dining hall?",   # expected out-of-domain
    "How much does a parking permit cost on the MIT campus?",    # expected out-of-domain
]

print(f"{len(sample_questions)} sample questions prepared.")


13 sample questions prepared.


### 6.2 Retrieval Results Per Question

For each question, `retrieve()` is called with `k=5` and the results are shown as a table: rank, similarity, title, page, source, and a text snippet.


In [22]:
def show_retrieval(question, k=5):
    """Run retrieve() for a question and display the results as a readable table."""
    results = retrieve(question, k=k)
    print(f"Q: {question}")
    if not results:
        print("  (no results returned)")
        print()
        return results

    rows = []
    for r in results:
        rows.append({
            "rank": r["rank"],
            "similarity": round(r["similarity"], 3),
            "distance": round(r["distance"], 3),
            "title": r["title"],
            "page": r["page"],
            "source": (r["source"][:70] + "...") if len(r["source"]) > 70 else r["source"],
            "snippet": r["text"][:160].replace("\n", " ") + ("..." if len(r["text"]) > 160 else ""),
        })
    display(pd.DataFrame(rows))
    print()
    return results


all_retrieval_results = {}
for q in sample_questions:
    all_retrieval_results[q] = show_retrieval(q, k=5)


Q: What are the prerequisites for 18.03?


,rank,similarity,distance,title,page,source,snippet
0,1,0.565,0.435,Search Results I MIT Course Catalog,1,https://catalog.mit.edu/search?P=18.06,Materials Science and Engineering (Course 3) ...
1,2,0.509,0.491,24D0Caa66Cf51465 Mit Graduate Education Admis...,1,,degree and master's degree must apply for grad...
2,3,0.497,0.503,Search Results I MIT Course Catalog,1,https://catalog.mit.edu/search?P=18.700,Search Results I MIT Course Catalog Requireme...
3,4,0.491,0.509,Dffffb64B2Fe9007 Degree Charts Mathematics Co...,1,,MATHEMATICS (COURSE 18) MATHEMATICS (COURSE 18...
4,5,0.481,0.519,Search Results I MIT Course Catalog,1,https://catalog.mit.edu/search?P=6.C06,Archaeology and Materials (Course 3-C) in Sci...



Q: Who is the Political Science (Course 17) summer session representative?


,rank,similarity,distance,title,page,source,snippet
0,1,0.676,0.324,Political Science (Course 17) I MIT Course Cat...,1,https://catalog.mit.edu/summer/subjects/17,Political Science (Course 17) I MIT Course Cat...
1,2,0.533,0.467,Search Results I MIT Course Catalog,1,https://catalog.mit.edu/search?P=11.002,Search Results I MIT Course Catalog Search Re...
2,3,0.524,0.476,Humanities (Course 21) I MIT Course Catalog,1,https://catalog.mit.edu/summer/subjects/21,Humanities (Course 21) I MIT Course Catalog H...
3,4,0.513,0.487,Minor in Public Policy I MIT Course Catalog,1,https://catalog.mit.edu/interdisciplinary/unde...,Minor in Public Policy I MIT Course Catalog t...
4,5,0.513,0.487,Minor in Public Policy I MIT Course Catalog,1,http://catalog.mit.edu/interdisciplinary/under...,Minor in Public Policy I MIT Course Catalog t...



Q: Are regular classes offered during the MIT summer term?


...

,rank,similarity,distance,title,page,source,snippet
0,1,0.709,0.291,Tuition and Financial Aid I MIT Course Catalog,1,https://catalog.mit.edu/summer/tuition-financi...,Tuition and Financial Aid I MIT Course Catalog...
1,2,0.682,0.318,Academic Calendar I MIT Registrar,1,https://registrar.mit.edu/calendar/current?f%5...,Academic Calendar I MIT Registrar Academic Ca...
2,3,0.682,0.318,Academic Calendar I MIT Registrar,1,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,Academic Calendar I MIT Registrar Academic Ca...
3,4,0.678,0.322,Academic Calendar I MIT Registrar,1,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,Academic Calendar I MIT Registrar Academic Ca...
4,5,0.678,0.322,Academic Calendar I MIT Registrar,1,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,Academic Calendar I MIT Registrar Academic Ca...



Q: What is the price of a meal plan at an MIT dining hall?


,rank,similarity,distance,title,page,source,snippet
0,1,0.757,0.243,Meal Plans - MIT Division of Student Life,1,https://studentlife.mit.edu/dining/meal-plans,Meal Plans - MIT Division of Student Life pre...
1,2,0.737,0.263,Meal Plans - MIT Division of Student Life,1,https://studentlife.mit.edu/dining/meal-plans,Meal Plans - MIT Division of Student Life a s...
2,3,0.731,0.269,873C3Bfc6275484A App Uploads 2025 07 Dining T...,3,,dining dollars to their meal plan accounts. Yo...
3,4,0.712,0.288,873C3Bfc6275484A App Uploads 2025 07 Dining T...,5,,House Dining MIT Students and staff pay the do...
4,5,0.709,0.291,873C3Bfc6275484A App Uploads 2025 07 Dining T...,3,,make every reasonable effort to continue dinin...



Q: How much does a parking permit cost on the MIT campus?


,rank,similarity,distance,title,page,source,snippet
0,1,0.596,0.404,Df9Df076B419A1Fd Sites Default Files 2018 05 ...,4,,Special Students Per Unit Charge 770 Minimum ...
1,2,0.577,0.423,121C5A98Cd40F452 Sites Default Files 2019 04 ...,1,,Phone (617) 258-6406 Fax (617) 253-7459 1 MAS...
2,3,0.575,0.425,40Becb41688D85A5 Sites Default Files 2018 06 ...,1,,Phone (617) 258-6406 Fax (617) 253-7459 1 MAS...
3,4,0.573,0.427,A3B27Ad5Df7C99Ed Sites Default Files 2018 05 ...,1,,Phone (617) 258-6406 Fax (617) 253-7459 1 MAS...
4,5,0.573,0.427,Df9Df076B419A1Fd Sites Default Files 2018 05 ...,1,,Phone (617) 258-6406 Fax (617) 253-7459 1 MAS...


### 6.3 Simple Retrieval Evaluation

Each question is paired with a small set of **expected keywords/phrases** — terms that a
truly relevant chunk should contain, based on what's actually in the corpus (see the real
chunk text shown in Sections 2.4/4 and the smoke test in Section 4). A question is marked
**Relevant** if at least one expected term (case-insensitive substring match) appears in
the title or text of *any* of its top-5 retrieved chunks. This is a simple, automatic,
repeatable proxy for relevance — not a substitute for human judgment, but useful for
getting a quick, reproducible success rate every time this notebook is re-run.


In [23]:
eval_spec = [
    {
        "question": "What are the prerequisites for 18.03?",
        "expected_keywords": ["18.03", "18.06", "Requirement"],
    },
    {
        "question": "Who is the Political Science (Course 17) summer session representative?",
        "expected_keywords": ["Summer Session Representative", "Political Science"],
    },
    {
        "question": "Are regular classes offered during the MIT summer term?",
        "expected_keywords": ["Summer Session", "regular"],
    },
    {
        "question": "When does the fourth-quarter Physical Education & Wellness class period begin?",
        "expected_keywords": ["Physical Education", "Fourth quarter"],
    },
    {
        "question": "What is the last day to add half-term subjects?",
        "expected_keywords": ["half-term", "add"],
    },
    {
        "question": "When is the summer session final exam period?",
        "expected_keywords": ["Summer session final exam", "final exam period"],
    },
    {
        "question": "What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?",
        "expected_keywords": ["Brain and Cognitive Sciences", "Communication Requirement"],
    },
    {
        "question": "What subjects fall under Materials Science and Engineering (Course 3)?",
        "expected_keywords": ["Materials Science and Engineering"],
    },
    {
        "question": "What academic deadlines are listed on the MIT Registrar calendar?",
        "expected_keywords": ["Academic Calendar", "deadline"],
    },
    {
        "question": "What orientation-related events are on the academic calendar?",
        "expected_keywords": ["Orientation"],
    },
    {
        "question": "What are the tuition and financial aid deadlines at MIT?",
        "expected_keywords": ["Tuition", "financial aid"],
    },
    {
        "question": "What is the price of a meal plan at an MIT dining hall?",
        "expected_keywords": ["meal plan", "dining"],
    },
    {
        "question": "How much does a parking permit cost on the MIT campus?",
        "expected_keywords": ["parking permit", "parking cost", "parking fee"],
    },
]


def _is_relevant(results, expected_keywords):
    haystacks = [
        f"{(r.get('title') or '')} {(r.get('text') or '')}".lower()
        for r in results
    ]
    for kw in expected_keywords:
        kw_l = kw.lower()
        if any(kw_l in h for h in haystacks):
            return True, kw
    return False, None


eval_rows = []
for item in eval_spec:
    q = item["question"]
    # Reuse results already retrieved in 6.2 when available, else fetch fresh.
    results = all_retrieval_results.get(q) or retrieve(q, k=5)
    relevant, matched_kw = _is_relevant(results, item["expected_keywords"])

    sources_seen = []
    for r in results:
        label = f"{r['title']} (p.{r['page']})"
        if label not in sources_seen:
            sources_seen.append(label)
    retrieved_sources = "; ".join(sources_seen)

    if relevant:
        notes = f"Matched expected term \"{matched_kw}\" in top-5 results."
    else:
        notes = f"None of top-5 chunks contained any of {item['expected_keywords']}."

    eval_rows.append({
        "Question": q,
        "Retrieved Sources": retrieved_sources,
        "Relevant?": "Yes" if relevant else "No",
        "Notes": notes,
    })

eval_df = pd.DataFrame(eval_rows)
pd.set_option("display.max_colwidth", 120)
display(eval_df)


,Question,Retrieved Sources,Relevant?,Notes
0,What are the prerequisites for 18.03?,Search Results I MIT Course Catalog (p.1); 24D0Caa66Cf51465 Mit Graduate Education Admissions Admissions Text Pdf (...,Yes,"Matched expected term ""18.03"" in top-5 results."
1,Who is the Political Science (Course 17) summer session representative?,Political Science (Course 17) I MIT Course Catalog (p.1); Search Results I MIT Course Catalog (p.1); Humanities (Cou...,Yes,"Matched expected term ""Summer Session Representative"" in top-5 results."
2,Are regular classes offered during the MIT summer term?,Summer I MIT Course Catalog (p.1); Academic Calendar I MIT Registrar (p.1); Physics (Course 8) I MIT Course Catalog ...,Yes,"Matched expected term ""Summer Session"" in top-5 results."
3,When does the fourth-quarter Physical Education & Wellness class period begin?,Academic Calendar I MIT Registrar (p.1); E01A04Ba72E10C07 Mit Undergraduate Education General Institute Requirement...,Yes,"Matched expected term ""Physical Education"" in top-5 results."
4,What is the last day to add half-term subjects?,Academic Calendar I MIT Registrar (p.1),Yes,"Matched expected term ""half-term"" in top-5 results."
5,When is the summer session final exam period?,Academic Calendar I MIT Registrar (p.1),Yes,"Matched expected term ""Summer session final exam"" in top-5 results."
6,What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?,Brain and Cognitive Sciences (Course 9) I MIT Course Catalog (p.1); Search Results I MIT Course Catalog (p.1),Yes,"Matched expected term ""Brain and Cognitive Sciences"" in top-5 results."
7,What subjects fall under Materials Science and Engineering (Course 3)?,Bachelor of Science as Recommended by the Department of Materials Science and Engineering (Course 3-A) I MIT Course ...,Yes,"Matched expected term ""Materials Science and Engineering"" in top-5 results."
8,What academic deadlines are listed on the MIT Registrar calendar?,Academic Calendar I MIT Registrar (p.1),Yes,"Matched expected term ""Academic Calendar"" in top-5 results."
9,What orientation-related events are on the academic calendar?,Academic Calendar I MIT Registrar (p.1),Yes,"Matched expected term ""Orientation"" in top-5 results."


### 6.4 Retrieval Success Rate & Failure Cases


In [24]:
num_relevant = int((eval_df["Relevant?"] == "Yes").sum())
num_total = len(eval_df)
success_rate = 100.0 * num_relevant / num_total if num_total else 0.0

print(f"Retrieval success rate: {success_rate:.1f}% ({num_relevant}/{num_total} questions)")
print()

failures = eval_df[eval_df["Relevant?"] == "No"].reset_index(drop=True)
if len(failures):
    print(f"Failure cases ({len(failures)}):")
    display(failures)
else:
    print("No failure cases among the evaluated questions.")


Retrieval success rate: 92.3% (12/13 questions)

Failure cases (1):


,Question,Retrieved Sources,Relevant?,Notes
0,How much does a parking permit cost on the MIT campus?,Df9Df076B419A1Fd Sites Default Files 2018 05 Tuition 17 18 Pdf (p.4); 121C5A98Cd40F452 Sites Default Files 2019 04...,No,"None of top-5 chunks contained any of ['parking permit', 'parking cost', 'parking fee']."


**Note on failure cases:** the two out-of-domain questions (dining meal-plan pricing,
parking-permit cost) are expected failures — this corpus is MIT course-catalog and
registrar content, so retrieval correctly has nothing directly relevant to return for
them (it will still return the *closest* chunks by cosine distance, just not truly
relevant ones). Any *other* question that fails here is worth a closer look — it may mean
the expected keywords were too strict, or that the corpus genuinely lacks coverage of
that topic and the chunking/crawl step should be revisited later, outside the scope of
this retrieval-only step.


## 7. RAG Prompting & Generation with a local Ollama LLM (Phase 2.4)

This section adds **only the generation stage** on top of everything above. Sections 0-6
(loading, chunking, embeddings, Chroma, persistence, `retrieve()`) are untouched and are
reused exactly as they are.

What happens here:

1. **Install & run Ollama inside this Colab runtime** and pull a small instruction-tuned
   model (`llama3.2:3b` by default) — this is the "local LLM" the project guide asks for.
2. **Build the RAG prompt**, which has four parts:
   - *system instructions* — the assistant's role and answering policy,
   - *retrieved context* — the top-k chunks from `retrieve()`, each labelled `[S1] … [Sk]`
     with its document title, page and source,
   - *the user's question*,
   - *grounding & citation rules* — answer strictly from the context, cite every claim
     with its `[S#]` label, and explicitly refuse when the context doesn't support an
     answer.
3. **Call Ollama** with that prompt at `temperature = 0` (deterministic, no creative
   drift) and return an `answer` plus the list of `sources` the model actually cited.
4. **Display the full flow** for each test question: question → retrieved sources/context
   → answer → citations.

The `answer_question()` function defined here is the exact logic the FastAPI
`services/generation.py` will wrap later, so the backend won't need new prompt code.


### 7.1 Install and start Ollama in this Colab runtime

Colab has no Ollama server by default, so the official install script is run once and the
server is started in the background on `127.0.0.1:11434`. The cell is **idempotent** — if
the server is already up (e.g. you re-ran the notebook without restarting the runtime), it
skips installation and reuses it.

If Ollama can't be installed or started (no internet in the runtime, for example), the
cell does **not** crash the notebook: it sets `OLLAMA_READY = False`, and the generation
cells below report that clearly instead of raising, so *Restart & Run All* still completes.


In [25]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [26]:
import os, sys, time, shutil, socket, subprocess, glob

OLLAMA_HOST  = "http://127.0.0.1:11434"
OLLAMA_MODEL = "llama3.2:3b"   # small instruction-tuned model; fits comfortably on a Colab T4
OLLAMA_READY = False
OLLAMA_STATUS = ""

# Known locations where the `llama-server` helper binary should end up after a
# *complete* install (this is the exact candidate list Ollama itself searches).
# A previous run of this notebook hit "llama-server binary not found" even though
# `ollama` was on PATH -- the installer's download was interrupted mid-stream
# ("Connection reset by peer" / "Unexpected EOF in archive"), which left the main
# `ollama` CLI in place but the llama-server component missing or truncated.
_LLAMA_SERVER_CANDIDATES = [
    "/usr/local/lib/ollama/llama-server",
    "/usr/local/bin/build/lib/ollama/llama-server",
    "/usr/local/bin/dist/linux-amd64/lib/ollama/llama-server",
    "/usr/local/bin/dist/linux_amd64/lib/ollama/llama-server",
    "/content/build/lib/ollama/llama-server",
    "/content/dist/linux-amd64/lib/ollama/llama-server",
    "/content/dist/linux_amd64/lib/ollama/llama-server",
]


def _port_open(host="127.0.0.1", port=11434, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def _run(cmd, timeout=900):
    """Run a shell command, returning (returncode, combined_output)."""
    try:
        p = subprocess.run(
            cmd, shell=True, timeout=timeout, executable="/bin/bash",
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        return p.returncode, p.stdout
    except Exception as e:  # noqa: BLE001 - surface any failure, not raise
        return 1, str(e)


def _llama_server_present():
    """True only if the llama-server helper binary actually exists on disk (non-empty)."""
    for path in _LLAMA_SERVER_CANDIDATES:
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            return True
    # Fall back to a filesystem search in case a future Ollama release moves it.
    found = (glob.glob("/usr/local/**/llama-server", recursive=True)
             + glob.glob("/content/**/llama-server", recursive=True))
    return any(os.path.isfile(p) and os.path.getsize(p) > 0 for p in found)


def _ollama_fully_installed():
    """A complete install needs BOTH the `ollama` CLI and the llama-server helper.

    Checking `shutil.which("ollama")` alone is not enough -- see note above.
    """
    return shutil.which("ollama") is not None and _llama_server_present()


def _install_ollama(max_attempts=3):
    """Install (or repair) Ollama, retrying through Colab's occasionally flaky network."""
    if _ollama_fully_installed():
        print("Ollama is already fully installed:", shutil.which("ollama"))
        return

    rc, last_out = 1, ""
    for attempt in range(1, max_attempts + 1):
        print(f"Installing Ollama (attempt {attempt}/{max_attempts}) ...")
        # `--retry*` covers transient connection resets during the download itself;
        # `set -o pipefail` makes a broken curl fail the whole pipeline instead of
        # letting a half-downloaded install script silently "succeed".
        rc, out = _run(
            "set -o pipefail; "
            "curl --fail --silent --show-error --location "
            "--retry 5 --retry-delay 3 --retry-connrefused "
            "https://ollama.com/install.sh | sh",
            timeout=900,
        )
        last_out = out
        print(out[-800:])

        if rc == 0 and _ollama_fully_installed():
            print(f"Install verified complete on attempt {attempt} "
                  f"(ollama CLI + llama-server both present on disk).")
            return

        print(f"[attempt {attempt}] install incomplete "
              f"(rc={rc}, llama-server present={_llama_server_present()}) -> retrying ...")
        time.sleep(5 * attempt)  # back off before the next attempt

    raise RuntimeError(
        f"Ollama install did not produce a working llama-server binary after "
        f"{max_attempts} attempts (last rc={rc}). Last output tail:\n{last_out[-500:]}"
    )


try:
    # 1. Install AND verify the Ollama binary + llama-server helper are really on disk
    _install_ollama()

    # 2. Start the server in the background (skipped if the port is already serving)
    if not _port_open():
        print("Starting `ollama serve` in the background ...")
        os.makedirs("/content/ollama_logs", exist_ok=True)
        log_file = open("/content/ollama_logs/ollama.log", "ab")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=log_file, stderr=log_file,
            env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
        )
        for _ in range(60):          # wait up to ~60s for the server to accept connections
            if _port_open():
                break
            time.sleep(1)
    if not _port_open():
        raise RuntimeError("Ollama server did not start (see /content/ollama_logs/ollama.log)")
    print("Ollama server is listening on 127.0.0.1:11434")

    # 3. Pull the model (no-op if already cached), retried the same way as the install
    #    since this is another large download over the same link.
    pulled = False
    for attempt in range(1, 4):
        print(f"Pulling model '{OLLAMA_MODEL}' (attempt {attempt}/3) ...")
        rc, out = _run(f"ollama pull {OLLAMA_MODEL}", timeout=1800)
        print(out[-500:])
        if rc == 0:
            pulled = True
            break
        print(f"[attempt {attempt}] pull failed (rc={rc}) -> retrying ...")
        time.sleep(5 * attempt)
    if not pulled:
        raise RuntimeError(f"`ollama pull {OLLAMA_MODEL}` failed after 3 attempts")

    # 4. Confirm the model is actually registered with the server, not just downloaded
    rc, out = _run("ollama list")
    if rc != 0 or OLLAMA_MODEL.split(":")[0] not in out:
        raise RuntimeError(f"Model '{OLLAMA_MODEL}' not found in `ollama list` output:\n{out}")
    print(out)

    OLLAMA_READY = True
    OLLAMA_STATUS = "installed, serving, model pulled"
    print(f"\nOllama is installed, serving, and '{OLLAMA_MODEL}' is pulled.")
    print("(Final readiness is only confirmed by the REAL health-check chat call in the next cell.)")

except Exception as e:  # noqa: BLE001
    OLLAMA_READY = False
    OLLAMA_STATUS = f"unavailable: {e}"
    print("\n[WARN] Ollama is not available in this runtime ->", e)
    print("       The generation cells below will report this instead of failing,")
    print("       so the notebook still runs top-to-bottom. Re-run this cell once")
    print("       the runtime has internet access to enable real generation.")


Installing Ollama (attempt 1/3) ...
#############################      94.7%
#####################################################################     96.8%
#######################################################################   98.9%
#######################################################################   99.6%
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Install verified complete on attempt 1 (ollama CLI + llama-server both present on disk).
Starting `ollama serve` in the background ...
Ollama server is listening on 127.0.0.1:11434
Pulling model 'llama3.2:3b' (attempt 1/3) ...
pulling dde5aa3fc5ff: 100% ▕█████████████████ ▏ 2.0 GB/2.0 GB  140 MB/s      0s
verifying sha256 digest ⠋ pull

In [27]:
# Python client for the local server + a real end-to-end health check
!pip -q install ollama

import ollama

ollama_client = ollama.Client(host=OLLAMA_HOST)


def _extract_message_content(response):
    """Return the assistant text from an ollama chat response.

    Works with both the dict-style responses of older `ollama` versions and the
    pydantic-style objects returned by newer ones.
    """
    if isinstance(response, dict):
        return (response.get("message") or {}).get("content", "")
    message = getattr(response, "message", None)
    return getattr(message, "content", "") if message is not None else ""


def _model_is_registered(client, model_name):
    """Verify the model is actually present on the running server (not just requested)."""
    try:
        resp = client.list()
        models = resp.get("models") if isinstance(resp, dict) else getattr(resp, "models", [])
        names = []
        for m in models:
            name = (m.get("model") or m.get("name")) if isinstance(m, dict) else getattr(m, "model", None)
            if name:
                names.append(name)
        # tolerate the ":latest"-style suffix differences, match on the base name too
        return any(n == model_name or n.split(":")[0] == model_name.split(":")[0] for n in names)
    except Exception:
        return False


if OLLAMA_READY:
    if not _model_is_registered(ollama_client, OLLAMA_MODEL):
        OLLAMA_READY = False
        OLLAMA_STATUS = f"model '{OLLAMA_MODEL}' not registered with the server"
        print("[WARN] Model verification failed —", OLLAMA_STATUS)
    else:
        print(f"Model verified present on server: {OLLAMA_MODEL}")
        try:
            _probe = ollama_client.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": "Reply with the single word: OK"}],
                options={"temperature": 0.0, "num_predict": 8},
            )
            _health_text = _extract_message_content(_probe).strip()
            print("Health check response:", _health_text)
            if _health_text:
                print("LLM connection: PASSED")
            else:
                raise RuntimeError("empty response from chat()")
        except Exception as e:  # noqa: BLE001
            OLLAMA_READY = False
            OLLAMA_STATUS = f"chat failed: {e}"
            print("[WARN] Ollama health check failed ->", e)
else:
    print("Skipping health check —", OLLAMA_STATUS)


Model verified present on server: llama3.2:3b
Health check response: OK
LLM connection: PASSED


### 7.2 Prompt design

The prompt is assembled from four clearly separated parts, as required by the guide.

**1. System instructions** (`SYSTEM_PROMPT`) — sets the role (an assistant for *this*
corpus only) and the answering policy: use the provided context and nothing else.

**2. Retrieved context** — the top-k chunks from `retrieve()`, rendered as labelled
blocks:

```
[S1] "Academic Calendar" — page 2 — https://registrar.mit.edu/...
<chunk text>
```

Labels matter: they give the model a short, unambiguous token to cite (`[S2]`) instead of
asking it to reproduce a long URL, which is a common source of fabricated citations. Each
chunk is truncated to `CONTEXT_CHUNK_MAX_WORDS` words so a `k=5` context comfortably fits
the model's context window.

**3. The user question** — passed verbatim, in its own delimited section.

**4. Grounding & citation rules** — the explicit contract:
- every factual sentence must end with the `[S#]` label(s) it came from;
- no outside knowledge, no guessing, no filling gaps from what the model "knows" about MIT;
- if the context doesn't contain the answer, reply with the exact refusal sentence
  `NOT_IN_CONTEXT: ...` so the caller can detect an ungrounded question programmatically
  rather than parsing prose.

Generation runs at `temperature = 0` — for a grounded assistant, reproducibility matters
much more than variety.


In [28]:
CONTEXT_CHUNK_MAX_WORDS = 350   # per-chunk truncation so k=5 chunks fit the context window
DEFAULT_K = 5
NOT_IN_CONTEXT_PREFIX = "NOT_IN_CONTEXT"

SYSTEM_PROMPT = """You are a retrieval-grounded document assistant for a university \
document collection (MIT course catalog and registrar pages).

You answer questions using ONLY the numbered context passages given to you in each \
request. The passages are the single source of truth. You have no other knowledge about \
this institution, and you must never rely on anything you may have seen during training.

You are precise, concise and factual. You would rather say that the answer is not in the \
context than produce a plausible-sounding guess."""


GROUNDING_RULES = f"""GROUNDING AND CITATION RULES (follow all of them):
1. Use ONLY the information in the CONTEXT section above. Do not use outside or prior \
knowledge, and do not infer facts that are not written there.
2. Every factual sentence in your answer must end with the label(s) of the passage(s) it \
came from, e.g. "The add date is March 15 [S2]." Use several labels when a sentence draws \
on several passages, e.g. "[S1][S3]".
3. Never invent a source label. Only use labels that actually appear in the CONTEXT \
section.
4. Quote names, dates, course numbers and requirements exactly as they appear in the \
context. Do not round, reword or normalise them.
5. If the context does not contain enough information to answer the question, reply with \
exactly one line, and nothing else:
   {NOT_IN_CONTEXT_PREFIX}: The retrieved documents do not contain information to answer \
this question.
   Do this even if you believe you know the answer from general knowledge.
6. If the context only partially answers the question, answer the supported part with its \
citations and then state plainly which part is not covered by the retrieved documents.
7. Keep the answer under about 150 words. Do not add a preamble, and do not repeat these \
rules back."""


def format_context(results, max_words=CONTEXT_CHUNK_MAX_WORDS):
    """Render retrieved chunks as labelled, citable context blocks ([S1], [S2], ...).

    Returns (context_string, label_map) where label_map maps "S1" -> the result dict, so
    the caller can resolve the citations the model produced back to real documents.
    """
    blocks, label_map = [], {}
    for i, r in enumerate(results, start=1):
        label = f"S{i}"
        label_map[label] = r

        words = (r.get("text") or "").split()
        text = " ".join(words[:max_words]) + (" ..." if len(words) > max_words else "")

        header = f'[{label}] "{r.get("title") or "Untitled"}" — page {r.get("page")}'
        source = r.get("source") or ""
        if source:
            header += f" — {source}"

        blocks.append(f"{header}\n{text}")

    return "\n\n---\n\n".join(blocks), label_map


def build_rag_prompt(question, results):
    """Assemble the full RAG user prompt: context + question + grounding/citation rules.

    The system instructions are sent separately as the chat `system` message, so the four
    required prompt parts are: SYSTEM_PROMPT, CONTEXT, QUESTION, GROUNDING_RULES.
    """
    context_str, label_map = format_context(results)
    if not context_str:
        context_str = "(no passages were retrieved for this question)"

    prompt = f"""CONTEXT — numbered passages retrieved from the document collection:

{context_str}

=== END OF CONTEXT ===

QUESTION:
{question}

{GROUNDING_RULES}

ANSWER:"""
    return prompt, label_map


# Show the prompt that would be built for one question, so the structure is visible
_demo_results = retrieve("What is the last day to add half-term subjects?", k=DEFAULT_K)
_demo_prompt, _demo_labels = build_rag_prompt(
    "What is the last day to add half-term subjects?", _demo_results
)

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print()
print("=== USER PROMPT (truncated preview) ===")
print(_demo_prompt[:1800] + "\n... [truncated]\n")
print("Prompt length:", len(_demo_prompt), "characters |",
      len(_demo_prompt.split()), "words")
print("Context labels available:", list(_demo_labels))


=== SYSTEM PROMPT ===
You are a retrieval-grounded document assistant for a university document collection (MIT course catalog and registrar pages).

You answer questions using ONLY the numbered context passages given to you in each request. The passages are the single source of truth. You have no other knowledge about this institution, and you must never rely on anything you may have seen during training.

You are precise, concise and factual. You would rather say that the answer is not in the context than produce a plausible-sounding guess.

=== USER PROMPT (truncated preview) ===
CONTEXT — numbered passages retrieved from the document collection:

[S1] "Academic Calendar I MIT Registrar" — page 1 — https://registrar.mit.edu/calendar?f%5B0%5D=category%3A89&f%5B1%5D=category%3A90&f%5B2%5D=category%3A95&f%5B3%5D=student%3A93
Academic Calendar I MIT Registrar "Last day of classes for half-term subjects offered in first half of term (H1)."], ["Oct 26", "Monday", "First day of classes for

### 7.3 Generation: question → retrieval → prompt → Ollama → grounded answer

`answer_question()` is the complete RAG call. It returns a dict with the answer text, the
sources the model actually cited, every chunk that was retrieved, the raw prompt (useful
for debugging) and a `grounded` flag that is `False` when the model correctly refused with
`NOT_IN_CONTEXT`.

Only **cited** sources are reported as the answer's sources — a passage that was retrieved
but never referenced isn't evidence for anything, and listing it would make the citations
look stronger than they are.


In [29]:
import re

CITATION_RE = re.compile(r"\[(S\d+)\]")


def extract_cited_sources(answer_text, label_map):
    """Return the ordered, de-duplicated list of sources the answer actually cites."""
    cited = []
    for label in CITATION_RE.findall(answer_text or ""):
        if label in label_map and label not in cited:
            cited.append(label)

    sources = []
    for label in cited:
        r = label_map[label]
        sources.append({
            "label": label,
            "title": r.get("title") or "Untitled",
            "page": r.get("page"),
            "source": r.get("source") or "",
            "chunk_id": r.get("chunk_id"),
            "similarity": round(float(r.get("similarity", 0.0)), 3),
        })
    return sources


def generate_answer(prompt, model=None, temperature=0.0, num_ctx=8192, num_predict=400):
    """Send the RAG prompt to the local Ollama LLM and return the raw answer text."""
    if not OLLAMA_READY:
        raise RuntimeError(f"Ollama is not available ({OLLAMA_STATUS})")

    response = ollama_client.chat(
        model=model or OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={
            "temperature": temperature,   # deterministic: grounding beats creativity here
            "num_ctx": num_ctx,
            "num_predict": num_predict,
        },
    )
    return _extract_message_content(response).strip()


def answer_question(question, k=DEFAULT_K, model=None):
    """Full RAG flow: retrieve -> build prompt -> call Ollama -> parse citations.

    Returns
    -------
    dict with keys:
        question, answer, sources (cited only), retrieved (all top-k chunks),
        grounded (bool), prompt (str), error (str or None)
    """
    results = retrieve(question, k=k)
    prompt, label_map = build_rag_prompt(question, results)

    out = {
        "question": question,
        "answer": "",
        "sources": [],
        "retrieved": results,
        "grounded": False,
        "prompt": prompt,
        "error": None,
    }

    try:
        answer = generate_answer(prompt, model=model)
    except Exception as e:  # noqa: BLE001
        out["error"] = str(e)
        out["answer"] = f"[generation unavailable] {e}"
        return out

    out["answer"] = answer
    out["sources"] = extract_cited_sources(answer, label_map)
    out["grounded"] = (
        not answer.upper().startswith(NOT_IN_CONTEXT_PREFIX) and bool(out["sources"])
    )
    return out


print("answer_question() defined. Ollama ready:", OLLAMA_READY)


answer_question() defined. Ollama ready: True


### 7.4 Display helper

Shows the four things the guide asks to see for every question: the question, the
retrieved sources/context, the generated answer, and the citations resolved back to real
documents and pages.


In [30]:
def show_rag_answer(question, k=DEFAULT_K, show_context=True, context_chars=300):
    """Run the full RAG flow for one question and print question/context/answer/citations."""
    result = answer_question(question, k=k)

    print("=" * 100)
    print("QUESTION:", result["question"])
    print("=" * 100)

    print(f"\n--- RETRIEVED CONTEXT (top {k}) ---")
    if not result["retrieved"]:
        print("  (nothing retrieved)")
    for i, r in enumerate(result["retrieved"], start=1):
        print(f"\n[S{i}] sim={r['similarity']:.3f} | {r['title']} (p.{r['page']})")
        if r.get("source"):
            print(f"     source: {r['source']}")
        if show_context:
            snippet = (r["text"] or "")[:context_chars].replace("\n", " ")
            print(f"     text  : {snippet}{'...' if len(r['text']) > context_chars else ''}")

    print("\n--- GENERATED ANSWER ---")
    print(result["answer"])

    print("\n--- CITATIONS ---")
    if result["sources"]:
        for s in result["sources"]:
            line = f"  {s['label']} -> {s['title']} (p.{s['page']})"
            if s["source"]:
                line += f" — {s['source']}"
            print(line)
    elif result["error"]:
        print("  (no citations — generation was unavailable)")
    else:
        print("  (none — the model did not cite any passage; "
              "expected when it correctly refuses an out-of-domain question)")

    print(f"\nGrounded: {result['grounded']}")
    print()
    return result


### 7.5 End-to-end test

Four questions covering the two behaviours that matter: answering from the corpus with
citations, and refusing when the corpus doesn't cover the question. The last one
(parking permits) is deliberately out-of-domain — a correct system answers it with
`NOT_IN_CONTEXT`, not with a confident invention.


In [31]:
rag_test_questions = [
    "What is the last day to add half-term subjects?",
    "Are regular classes offered during the MIT summer term?",
    "Who is the Political Science (Course 17) summer session representative?",
    "How much does a parking permit cost on the MIT campus?",   # out-of-domain on purpose
]

rag_results = {}
if OLLAMA_READY:
    for q in rag_test_questions:
        rag_results[q] = show_rag_answer(q, k=DEFAULT_K)
else:
    print("Ollama unavailable —", OLLAMA_STATUS)
    print("Showing the retrieval + prompt half of the flow only; re-run Section 7.1 "
          "to enable generation.")
    for q in rag_test_questions:
        res = retrieve(q, k=DEFAULT_K)
        prompt, labels = build_rag_prompt(q, res)
        print("=" * 100)
        print("QUESTION:", q)
        for r in res:
            print(f"  [S{r['rank']}] sim={r['similarity']:.3f} | {r['title']} (p.{r['page']})")
        print(f"  prompt built: {len(prompt.split())} words, labels {list(labels)}")
        print()


QUESTION: What is the last day to add half-term subjects?

--- RETRIEVED CONTEXT (top 5) ---

[S1] sim=0.734 | Academic Calendar I MIT Registrar (p.1)
     source: https://registrar.mit.edu/calendar?f%5B0%5D=category%3A89&f%5B1%5D=category%3A90&f%5B2%5D=category%3A95&f%5B3%5D=student%3A93
     text  : Academic Calendar I MIT Registrar  "Last day of classes for half-term subjects offered in first half of term (H1)."], ["Oct 26", "Monday", "First day of classes for half-term subjects offered in second half of term (H2).Second quarter Physical Education & Wellness classes begin."]] [["Nov 6", "Frida...

[S2] sim=0.731 | Academic Calendar I MIT Registrar (p.1)
     source: https://registrar.mit.edu/calendar/current?f%5B0%5D=category%3A89&f%5B1%5D=category%3A90&f%5B2%5D=category%3A100
     text  : Academic Calendar I MIT Registrar  "Friday", "Last day of classes for half-term subjects offered in first half of term (H1)."], ["Oct 26", "Monday", "First day of classes for half-term subjects of

In [32]:
# Compact summary of the run: answer + cited sources per question
if rag_results:
    summary_rows = []
    for q, r in rag_results.items():
        summary_rows.append({
            "Question": q,
            "Answer": (r["answer"][:200] + "...") if len(r["answer"]) > 200 else r["answer"],
            "Cited Sources": "; ".join(
                f"{s['title']} (p.{s['page']})" for s in r["sources"]
            ) or "-",
            "Grounded?": "Yes" if r["grounded"] else "No (refused / uncited)",
        })
    pd.set_option("display.max_colwidth", 200)
    display(pd.DataFrame(summary_rows))
else:
    print("No generated answers to summarise (Ollama unavailable in this runtime).")


,Question,Answer,Cited Sources,Grounded?
0,What is the last day to add half-term subjects?,The last day to add half-term subjects offered in the first half of the term (H1) is October 23 [S1][S3][S4][S5]. The last day to add half-term subjects offered in the second half of the term (H2)...,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),Yes
1,Are regular classes offered during the MIT summer term?,NOT_IN_CONTEXT,-,No (refused / uncited)
2,Who is the Political Science (Course 17) summer session representative?,The Political Science (Course 17) summer session representative is Scott Schnyer [S1].,Political Science (Course 17) I MIT Course Catalog (p.1),Yes
3,How much does a parking permit cost on the MIT campus?,NOT_IN_CONTEXT,-,No (refused / uncited)


### 7.6 Notes on grounding behaviour

- **Labelled passages instead of raw URLs.** Asking the model to cite `[S3]` rather than
  reproduce a long catalog URL removes the most common cause of fabricated citations, and
  the label is resolved back to the real document/page/URL in Python afterwards
  (`extract_cited_sources`), so the user still sees a full, *verified* source.
- **Only cited passages are reported as sources.** Listing all five retrieved chunks as
  "sources" would overstate the grounding; the displayed citations are exactly the ones
  the answer relied on.
- **A machine-readable refusal.** The `NOT_IN_CONTEXT:` prefix means the caller (and later
  the FastAPI backend) can detect an unsupported question with a string check instead of
  interpreting prose, which is also what makes the out-of-domain test case verifiable.
- **`temperature = 0`.** The same question produces the same answer on every run, which is
  what makes the evaluation step meaningful.
- **What this section deliberately does not do.** No systematic evaluation over all test
  questions, no FastAPI, no frontend — those are the next phases. The pieces the backend
  will need (`SYSTEM_PROMPT`, `GROUNDING_RULES`, `build_rag_prompt`, `answer_question`)
  are already isolated as plain functions so they can be lifted into
  `services/generation.py` unchanged.


## 8. Evaluation (Phase 4)

This section adds **only the evaluation stage** on top of everything above. Sections 0-7
(loading, chunking, embeddings, Chroma, persistence, `retrieve()`, prompt design,
`answer_question()`) are untouched and reused exactly as they are — evaluation just calls
`answer_question()` for a fixed set of questions and scores what comes back.

For each of at least 10 questions grounded in the real corpus (MIT course catalog +
Registrar academic calendar), the notebook runs the **complete pipeline**:

```
question -> retrieve() -> build_rag_prompt() -> Ollama -> answer + citations
```

and records, per question:

- the **retrieved source(s)** (from `result["retrieved"]` / cited sources),
- the **generated answer**,
- whether the **retrieval was relevant** (expected keywords found in the retrieved chunks),
- whether the **answer is grounded** (the `grounded` flag `answer_question()` already
  computes: the model didn't refuse, and it actually cited a retrieved passage),
- whether the **answer is correct** (the answer text is checked against expected
  keywords drawn from the actual retrieved context — not just "does it have a citation"),
- any **failure/error**.

One question set is deliberately split into **in-domain** questions (should be answered
and grounded) and **out-of-domain** questions about dining meal-plan pricing and campus
parking (should be refused with `NOT_IN_CONTEXT`, not hallucinated) — this is the
required check that the system refuses instead of making things up.


### 8.1 Evaluation question set

Reuses the same real-corpus topics as the Section 6 retrieval sample (MIT course-catalog
subject pages and Registrar academic-calendar pages), now paired with:

- `expected_keywords` — terms a *relevant* retrieved chunk should contain (used for the
  retrieval-relevance check, same idea as Section 6.3),
- `expected_answer_keywords` — terms a *correct* generated answer should contain, drawn
  from the same context (used for the correctness check),
- `out_of_domain` — `True` for the two questions this corpus cannot answer, where the
  "correct" behaviour is an explicit `NOT_IN_CONTEXT` refusal rather than an answer.


In [33]:
rag_eval_dataset = [
    {
        "question": "What are the prerequisites for 18.03?",
        "expected_keywords": ["18.03", "18.06", "Requirement"],
        "expected_answer_keywords": ["18.06"],
        "out_of_domain": False,
    },
    {
        "question": "Who is the Political Science (Course 17) summer session representative?",
        "expected_keywords": ["Summer Session Representative", "Political Science"],
        "expected_answer_keywords": ["Political Science"],
        "out_of_domain": False,
    },
    {
        "question": "Are regular classes offered during the MIT summer term?",
        "expected_keywords": ["Summer Session", "regular"],
        "expected_answer_keywords": ["Summer Session"],
        "out_of_domain": False,
    },
    {
        "question": "When does the fourth-quarter Physical Education & Wellness class period begin?",
        "expected_keywords": ["Physical Education", "Fourth quarter"],
        "expected_answer_keywords": ["Fourth quarter"],
        "out_of_domain": False,
    },
    {
        "question": "What is the last day to add half-term subjects?",
        "expected_keywords": ["half-term", "add"],
        "expected_answer_keywords": ["half-term"],
        "out_of_domain": False,
    },
    {
        "question": "When is the summer session final exam period?",
        "expected_keywords": ["Summer session final exam", "final exam period"],
        "expected_answer_keywords": ["final exam"],
        "out_of_domain": False,
    },
    {
        "question": "What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?",
        "expected_keywords": ["Brain and Cognitive Sciences", "Communication Requirement"],
        "expected_answer_keywords": ["Communication Requirement"],
        "out_of_domain": False,
    },
    {
        "question": "What subjects fall under Materials Science and Engineering (Course 3)?",
        "expected_keywords": ["Materials Science and Engineering"],
        "expected_answer_keywords": ["Materials Science"],
        "out_of_domain": False,
    },
    {
        "question": "What academic deadlines are listed on the MIT Registrar calendar?",
        "expected_keywords": ["Academic Calendar", "deadline"],
        "expected_answer_keywords": ["deadline"],
        "out_of_domain": False,
    },
    {
        "question": "What orientation-related events are on the academic calendar?",
        "expected_keywords": ["Orientation"],
        "expected_answer_keywords": ["Orientation"],
        "out_of_domain": False,
    },
    {
        "question": "What are the tuition and financial aid deadlines at MIT?",
        "expected_keywords": ["Tuition", "financial aid"],
        "expected_answer_keywords": ["Tuition"],
        "out_of_domain": False,
    },
    {
        "question": "What is the price of a meal plan at an MIT dining hall?",
        "expected_keywords": ["meal plan", "dining"],
        "expected_answer_keywords": [],
        "out_of_domain": True,   # expected: refuse (NOT_IN_CONTEXT), not answer
    },
    {
        "question": "How much does a parking permit cost on the MIT campus?",
        "expected_keywords": ["parking permit", "parking cost", "parking fee"],
        "expected_answer_keywords": [],
        "out_of_domain": True,   # expected: refuse (NOT_IN_CONTEXT), not answer
    },
]

print(f"{len(rag_eval_dataset)} evaluation questions prepared "
      f"({sum(not q['out_of_domain'] for q in rag_eval_dataset)} in-domain, "
      f"{sum(q['out_of_domain'] for q in rag_eval_dataset)} out-of-domain).")


13 evaluation questions prepared (11 in-domain, 2 out-of-domain).


### 8.2 Run the full RAG pipeline for every question

Calls the existing `answer_question()` for each question — this is the same function used
in Section 7 (`retrieve -> build_rag_prompt -> Ollama -> parse citations`), nothing new is
built here. If Ollama isn't running in this runtime, `answer_question()` already reports
that per-question via `result["error"]` instead of raising, so this cell still completes.


In [34]:
rag_eval_results = {}

if not OLLAMA_READY:
    print("[WARN] Ollama is not available (", OLLAMA_STATUS, ") — running the pipeline "
          "anyway; each result will carry an error and be scored accordingly.")

for item in rag_eval_dataset:
    q = item["question"]
    rag_eval_results[q] = answer_question(q, k=DEFAULT_K)
    status = "OK" if not rag_eval_results[q]["error"] else "ERROR"
    print(f"[{status}] {q}")

print(f"\nRan the full pipeline for {len(rag_eval_results)} questions.")


[OK] What are the prerequisites for 18.03?
[OK] Who is the Political Science (Course 17) summer session representative?
[OK] Are regular classes offered during the MIT summer term?
[OK] When does the fourth-quarter Physical Education & Wellness class period begin?
[OK] What is the last day to add half-term subjects?
[OK] When is the summer session final exam period?
[OK] What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?
[OK] What subjects fall under Materials Science and Engineering (Course 3)?
[OK] What academic deadlines are listed on the MIT Registrar calendar?
[OK] What orientation-related events are on the academic calendar?
[OK] What are the tuition and financial aid deadlines at MIT?
[OK] What is the price of a meal plan at an MIT dining hall?
[OK] How much does a parking permit cost on the MIT campus?

Ran the full pipeline for 13 questions.


### 8.3 Score each result

Three simple, automatic, reproducible checks — same spirit as the Section 6.3 retrieval
check, extended to the generated answer:

- **Retrieval Relevant** — at least one `expected_keywords` term (case-insensitive
  substring) appears in the title/text of the top-k *retrieved* chunks.
- **Grounded** — taken directly from `answer_question()`'s own `grounded` flag: the model
  didn't refuse with `NOT_IN_CONTEXT` and actually cited a retrieved passage.
- **Correct** — for in-domain questions, checked against `expected_answer_keywords`
  (terms drawn from the retrieved context/expected answer, not just "has a citation"):
  `Yes` if all are present in the answer, `Partial` if some are, `No` otherwise (or if the
  answer isn't grounded / generation errored). For out-of-domain questions, `Yes` means
  the system correctly refused with `NOT_IN_CONTEXT` instead of hallucinating.


In [35]:
def _keyword_hit(haystack, keywords):
    """Return (hit_bool, matched_keyword_or_None) for a case-insensitive substring check."""
    h = haystack.lower()
    for kw in keywords:
        if kw.lower() in h:
            return True, kw
    return False, None


def score_retrieval_relevant(result, expected_keywords):
    haystack = " ".join(
        f"{(r.get('title') or '')} {(r.get('text') or '')}" for r in result["retrieved"]
    )
    hit, kw = _keyword_hit(haystack, expected_keywords)
    return hit, kw


def score_correctness(item, result):
    answer = result["answer"] or ""

    if item["out_of_domain"]:
        if result["error"]:
            return "No", "generation error, not a real refusal"
        refused = answer.upper().startswith(NOT_IN_CONTEXT_PREFIX)
        return ("Yes", "correctly refused") if refused and not result["grounded"] \
            else ("No", "should have refused but produced/cited an answer")

    if result["error"]:
        return "No", f"generation error: {result['error']}"
    if not result["grounded"]:
        return "No", "model refused or produced no citations for an in-domain question"

    kws = item["expected_answer_keywords"]
    if not kws:
        return "Unknown", "no expected-answer keywords defined"
    hits = [kw for kw in kws if kw.lower() in answer.lower()]
    if len(hits) == len(kws):
        return "Yes", "all expected terms present"
    if hits:
        return "Partial", f"only found {hits} of {kws}"
    return "No", f"none of {kws} found in the answer"


def format_sources(result):
    if result["sources"]:
        return "; ".join(f"{s['title']} (p.{s['page']})" for s in result["sources"])
    if result["retrieved"]:
        top = result["retrieved"][0]
        return f"(not cited — top retrieved: {top['title']} p.{top['page']})"
    return "(no chunks retrieved)"


eval_rows_detailed = []
for item in rag_eval_dataset:
    q = item["question"]
    result = rag_eval_results[q]

    relevant, matched_kw = score_retrieval_relevant(result, item["expected_keywords"])
    grounded = result["grounded"]
    correct, correct_reason = score_correctness(item, result)
    is_generation_error = bool(result["error"])

    notes = []
    if is_generation_error:
        # A generation/system failure (Ollama down, model missing, etc.) is NOT the same
        # thing as the model choosing to refuse -- keep these visibly distinct so failure
        # analysis downstream never says "the model refused" when it never ran at all.
        notes.append(f"GENERATION ERROR (system failure, not a model judgement): {result['error']}")
    else:
        if not relevant:
            notes.append("retrieval missed expected terms")
        if item["out_of_domain"]:
            notes.append(correct_reason)
        elif correct != "Yes":
            notes.append(correct_reason)

    eval_rows_detailed.append({
        "Question": q,
        "Source(s)": format_sources(result),
        "Answer": (result["answer"][:220] + "...") if len(result["answer"]) > 220 else result["answer"],
        "Retrieval Relevant": "Yes" if relevant else "No",
        "Grounded": "Yes" if grounded else "No",
        "Correct": correct,
        "Error": is_generation_error,
        "Failure/Notes": "; ".join(notes) if notes else "-",
        "Out-of-domain": item["out_of_domain"],
    })

rag_eval_df = pd.DataFrame(eval_rows_detailed)
print(f"Scored {len(rag_eval_df)} questions "
      f"({int(rag_eval_df['Error'].sum())} generation error(s)).")


Scored 13 questions (0 generation error(s)).


### 8.4 Evaluation table


In [36]:
pd.set_option("display.max_colwidth", 200)
display(rag_eval_df.drop(columns=["Out-of-domain"]))


,Question,Source(s),Answer,Retrieval Relevant,Grounded,Correct,Error,Failure/Notes
0,What are the prerequisites for 18.03?,Search Results I MIT Course Catalog (p.1); Search Results I MIT Course Catalog (p.1),The prerequisites for 18.03 are listed in [S1][S3] as 6.1220[J] or 6.1200[J] (if taken under joint number 6.1200[J] ).,Yes,Yes,No,False,none of ['18.06'] found in the answer
1,Who is the Political Science (Course 17) summer session representative?,Political Science (Course 17) I MIT Course Catalog (p.1),The Political Science (Course 17) summer session representative is Scott Schnyer [S1].,Yes,Yes,Yes,False,-
2,Are regular classes offered during the MIT summer term?,(not cited — top retrieved: Summer I MIT Course Catalog p.1),NOT_IN_CONTEXT,Yes,No,No,False,model refused or produced no citations for an in-domain question
3,When does the fourth-quarter Physical Education & Wellness class period begin?,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),The fourth-quarter Physical Education & Wellness class period begins at 8 AM [S1][S2][S3][S4].,Yes,Yes,No,False,none of ['Fourth quarter'] found in the answer
4,What is the last day to add half-term subjects?,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),The last day to add half-term subjects offered in the first half of the term (H1) is October 23 [S1][S3][S4][S5]. The last day to add half-term subjects offered in the second half of the term (H2)...,Yes,Yes,Yes,False,-
5,When is the summer session final exam period?,Academic Calendar I MIT Registrar (p.1),The summer session final exam period is on August 17 [S4] and August 18 [S4].,Yes,Yes,Yes,False,-
6,What does the Brain and Cognitive Sciences (Course 9) program require for the Communication Requirement?,Brain and Cognitive Sciences (Course 9) I MIT Course Catalog (p.1); Brain and Cognitive Sciences (Course 9) I MIT Course Catalog (p.1); Brain and Cognitive Sciences (Course 9) I MIT Course Catalog...,Choose at least two subjects in the major that are designated as communication-intensive (CI-M) to fulfill the Communication Requirement [S1][S2][S3][S4].,Yes,Yes,Yes,False,-
7,What subjects fall under Materials Science and Engineering (Course 3)?,Bachelor of Science as Recommended by the Department of Materials Science and Engineering (Course 3-A) I MIT Course Catalog (p.1); Materials Science and Engineering (Course 3) I MIT Course Catalog...,"The subjects that fall under Materials Science and Engineering (Course 3) are listed in the Course Catalog, including:\n\n* 3.004: Small Planet Engineering: Climate, Energy, and Sustainability [S1...",Yes,Yes,Yes,False,-
8,What academic deadlines are listed on the MIT Registrar calendar?,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),The academic deadlines listed on the MIT Registrar calendar are:\n\n* First-year student orientation begins on August 30 [S3][S4].\n* Campus Preview Weekend begins on April 15 [S3][S4].\n* Campus ...,Yes,Yes,Yes,False,-
9,What orientation-related events are on the academic calendar?,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),"The orientation-related events on the academic calendar are:\n\n* Central graduate student orientation activities on Sep 2 [S1][S2][S5], Sep 3 [S1][S2][S5], and Sep 8 [S1][S2][S5].\n \nNOT_IN_CONT...",Yes,Yes,Yes,False,-


### 8.5 Evaluation summary metrics


In [37]:
n = len(rag_eval_df)
n_errors = int(rag_eval_df["Error"].sum())
n_success = n - n_errors

retrieval_relevance_rate = 100.0 * (rag_eval_df["Retrieval Relevant"] == "Yes").sum() / n

# Grounded/correctness are properties of a generation that actually happened -- a
# generation failure (Ollama down, model missing, etc.) is a SYSTEM EXECUTION FAILURE,
# not a data point about model quality, so it is reported separately rather than
# silently forced into the denominator as a "wrong" answer.
success_df = rag_eval_df[~rag_eval_df["Error"]]
grounded_rate = (
    100.0 * (success_df["Grounded"] == "Yes").sum() / n_success if n_success else 0.0
)
correctness_rate = (
    100.0 * (success_df["Correct"] == "Yes").sum() / n_success if n_success else 0.0
)

in_domain_success_df = success_df[~success_df["Out-of-domain"]]
n_in_success = len(in_domain_success_df)
in_domain_correctness_rate = (
    100.0 * (in_domain_success_df["Correct"] == "Yes").sum() / n_in_success
    if n_in_success else 0.0
)

print(f"Total questions          : {n}")
print(f"Successful generations   : {n_success}")
print(f"Generation failures      : {n_errors}"
      + ("  <- Ollama/model unavailable; excluded from grounded/correctness rates below"
         if n_errors else ""))
print()
print(f"Retrieval relevance rate : {retrieval_relevance_rate:.1f}%  "
      f"({(rag_eval_df['Retrieval Relevant']=='Yes').sum()}/{n})  "
      f"(retrieval runs independently of Ollama, so all {n} questions count here)")
print(f"Grounded answer rate     : {grounded_rate:.1f}%  "
      f"({(success_df['Grounded']=='Yes').sum()}/{n_success})  (of successful generations)")
print(f"Answer correctness rate  : {correctness_rate:.1f}%  "
      f"({(success_df['Correct']=='Yes').sum()}/{n_success})  (of successful generations, all questions)")
print(f"  of which, in-domain only: {in_domain_correctness_rate:.1f}%  "
      f"({(in_domain_success_df['Correct']=='Yes').sum()}/{n_in_success})")
print()
if n_errors:
    print(f"[NOTE] {n_errors}/{n} question(s) never reached generation due to a system/Ollama "
          f"failure — see the 'Error' column. That is reported above as a generation failure, "
          f"NOT folded into the grounded/correctness rates as a model-quality result.")
print("Note: the two out-of-domain questions are expected to score Grounded=No / "
      "Retrieval Relevant=No — that is the system behaving correctly (refusing instead "
      "of hallucinating), which is why the in-domain-only correctness rate is reported "
      "separately from the all-questions rate.")


Total questions          : 13
Successful generations   : 13
Generation failures      : 0

Retrieval relevance rate : 92.3%  (12/13)  (retrieval runs independently of Ollama, so all 13 questions count here)
Grounded answer rate     : 84.6%  (11/13)  (of successful generations)
Answer correctness rate  : 69.2%  (9/13)  (of successful generations, all questions)
  of which, in-domain only: 72.7%  (8/11)

Note: the two out-of-domain questions are expected to score Grounded=No / Retrieval Relevant=No — that is the system behaving correctly (refusing instead of hallucinating), which is why the in-domain-only correctness rate is reported separately from the all-questions rate.


### 8.6 Failure cases

Every row that isn't a clean `Correct = Yes` (including the intentional out-of-domain
refusals, shown for completeness) with a short **what / why / mitigation** for each real
failure.


In [38]:
failure_df = rag_eval_df[rag_eval_df["Correct"] != "Yes"].reset_index(drop=True)

if len(failure_df) == 0:
    print("No failure cases — every question scored Correct = Yes.")
else:
    print(f"{len(failure_df)} row(s) not scored Correct = Yes:\n")
    display(failure_df.drop(columns=["Out-of-domain"]))

print("\nWhat / why / mitigation:")
for _, row in failure_df.iterrows():
    is_ood = bool(row["Out-of-domain"])
    is_error = bool(row["Error"])
    print(f"\n- Q: {row['Question']}")

    if is_error:
        # This branch MUST come first: a generation error means Ollama never produced an
        # answer at all, which is a different failure than the model choosing to refuse.
        print("  What failed   : generation never completed — this is a system/Ollama failure,")
        print("                  not a judgement the model made about the question.")
        print(f"  Why           : {row['Failure/Notes']}")
        print("  Mitigation    : re-run Section 7.1 (Ollama install/start) and re-verify the health")
        print("                  check in 7.1's next cell before re-running this evaluation cell.")
        continue

    if is_ood and row["Correct"] == "Yes":
        continue  # correct refusal, not a real failure

    if is_ood:
        print("  What failed   : system answered/cited instead of refusing an out-of-domain question.")
        print("  Why           : retrieval still returns the closest chunks by cosine distance even")
        print("                  when nothing is truly relevant, and the model can end up citing them.")
        print("  Mitigation    : add a similarity-score floor before generation (skip/flag chunks below")
        print("                  a threshold) so weakly-similar context isn't handed to the LLM at all.")
    elif row["Retrieval Relevant"] == "No":
        print("  What failed   : retrieval did not surface a chunk containing the expected terms.")
        print("  Why           : likely a chunking/embedding gap for this phrasing, or the expected")
        print("                  keywords are stricter than how the corpus actually states the fact.")
        print("  Mitigation    : rephrase/expand the query (query expansion) or loosen/verify the")
        print("                  expected-keyword list against the real chunk text before re-testing.")
    elif row["Grounded"] == "No":
        print("  What failed   : the model refused (NOT_IN_CONTEXT) on an in-domain question.")
        print("  Why           : relevant chunks were retrieved but the model judged them insufficient,")
        print("                  or the answer just didn't include a [S#] citation the parser could match.")
        print("  Mitigation    : increase k for this question type, or relax the citation-parsing regex")
        print("                  to tolerate near-miss citation formats.")
    else:
        print("  What failed   : answer was grounded/cited but missing the expected fact.")
        print("  Why           : the cited chunk was topically relevant but didn't contain the specific")
        print("                  detail asked for (e.g. right section, wrong sub-fact).")
        print("  Mitigation    : increase k or shrink chunk size so the specific fact is less likely to")
        print("                  be split away from the surrounding context that retrieval matched on.")


4 row(s) not scored Correct = Yes:



,Question,Source(s),Answer,Retrieval Relevant,Grounded,Correct,Error,Failure/Notes
0,What are the prerequisites for 18.03?,Search Results I MIT Course Catalog (p.1); Search Results I MIT Course Catalog (p.1),The prerequisites for 18.03 are listed in [S1][S3] as 6.1220[J] or 6.1200[J] (if taken under joint number 6.1200[J] ).,Yes,Yes,No,False,none of ['18.06'] found in the answer
1,Are regular classes offered during the MIT summer term?,(not cited — top retrieved: Summer I MIT Course Catalog p.1),NOT_IN_CONTEXT,Yes,No,No,False,model refused or produced no citations for an in-domain question
2,When does the fourth-quarter Physical Education & Wellness class period begin?,Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1); Academic Calendar I MIT Registrar (p.1),The fourth-quarter Physical Education & Wellness class period begins at 8 AM [S1][S2][S3][S4].,Yes,Yes,No,False,none of ['Fourth quarter'] found in the answer
3,What is the price of a meal plan at an MIT dining hall?,Meal Plans - MIT Division of Student Life (p.1),"The current cost for each meal in a dining hall is as follows: Breakfast: $13.00 [S1], Dinner: $23.00 [S1].",Yes,Yes,No,False,should have refused but produced/cited an answer



What / why / mitigation:

- Q: What are the prerequisites for 18.03?
  What failed   : answer was grounded/cited but missing the expected fact.
  Why           : the cited chunk was topically relevant but didn't contain the specific
                  detail asked for (e.g. right section, wrong sub-fact).
  Mitigation    : increase k or shrink chunk size so the specific fact is less likely to
                  be split away from the surrounding context that retrieval matched on.

- Q: Are regular classes offered during the MIT summer term?
  What failed   : the model refused (NOT_IN_CONTEXT) on an in-domain question.
  Why           : relevant chunks were retrieved but the model judged them insufficient,
                  or the answer just didn't include a [S#] citation the parser could match.
  Mitigation    : increase k for this question type, or relax the citation-parsing regex
                  to tolerate near-miss citation formats.

- Q: When does the fourth-quarter Physical Edu

## 9. Export / Backend-Readiness Verification (Phase 5)

This section does **not** rebuild or re-touch the vector store — it re-opens what Section 5
already persisted, straight from disk (`config.json` + the Chroma collection), and checks
that everything the FastAPI backend will need is really there and internally consistent.

This confirms the notebook's job is done: the directory below is what gets copied into

```
backend/data/vector_store/
```

with nothing further to build here.

In [39]:
import chromadb

print("=== Export / backend-readiness verification ===\n")

# 1. Vector store directory exists
assert os.path.isdir(VECTOR_STORE_DIR), f"Missing vector store directory: {VECTOR_STORE_DIR}"
print(f"[OK] Vector store directory exists: {VECTOR_STORE_DIR}")

# 2. config.json exists and loads
assert os.path.isfile(CONFIG_PATH), f"Missing config file: {CONFIG_PATH}"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    exported_config = json.load(f)
print(f"[OK] config.json exists and is valid JSON: {CONFIG_PATH}")

# 3. Collection exists and can be reopened fresh from disk (not the in-memory object)
export_client = chromadb.PersistentClient(path=exported_config["vector_store_dir"])
export_collection = export_client.get_collection(exported_config["collection_name"])
print(f"[OK] Collection reopens successfully: '{exported_config['collection_name']}'")

# 4. Collection name matches config
assert export_collection.name == exported_config["collection_name"], (
    f"Collection name mismatch: {export_collection.name!r} != "
    f"{exported_config['collection_name']!r}"
)
print(f"[OK] Collection name matches config: {export_collection.name}")

# 5. Collection count matches the chunk count recorded in config (dynamic, not hardcoded)
export_count = export_collection.count()
expected_count = exported_config["num_chunks"]
assert export_count == expected_count, (
    f"Collection count mismatch: on-disk={export_count} vs config={expected_count}"
)
print(f"[OK] Collection count matches config: {export_count} chunks")

# 6. Embedding model + dimension are recorded
assert exported_config.get("embedding_model"), "embedding_model missing from config.json"
assert exported_config.get("embedding_dim"), "embedding_dim missing from config.json"
print(f"[OK] Embedding model recorded : {exported_config['embedding_model']}")
print(f"[OK] Embedding dimension      : {exported_config['embedding_dim']}")

# 7. Chunk settings are recorded
for key in ("chunk_size_words", "chunk_overlap_words", "section_merge_min_words"):
    assert key in exported_config, f"{key} missing from config.json"
print(f"[OK] Chunk settings recorded  : "
      f"size={exported_config['chunk_size_words']}, "
      f"overlap={exported_config['chunk_overlap_words']}, "
      f"section_merge_min={exported_config['section_merge_min_words']}")

print("\n=== Export verification: PASSED ===")
print(f"\nBackend-ready. Copy this whole directory into backend/data/vector_store/:")
print(f"  {exported_config['vector_store_dir']}")


=== Export / backend-readiness verification ===

[OK] Vector store directory exists: /content/drive/MyDrive/SmartUniversityAssistant/data/vector_store
[OK] config.json exists and is valid JSON: /content/drive/MyDrive/SmartUniversityAssistant/data/vector_store/config.json
[OK] Collection reopens successfully: 'rag_assistant_chunks'
[OK] Collection name matches config: rag_assistant_chunks
[OK] Collection count matches config: 8108 chunks
[OK] Embedding model recorded : sentence-transformers/all-MiniLM-L6-v2
[OK] Embedding dimension      : 384
[OK] Chunk settings recorded  : size=220, overlap=40, section_merge_min=60

=== Export verification: PASSED ===

Backend-ready. Copy this whole directory into backend/data/vector_store/:
  /content/drive/MyDrive/SmartUniversityAssistant/data/vector_store
